# Dice Dataset Generator

Generates a synthetic dataset of dice images with structured symbol sequences and natural language labels.

**Three-stage pipeline:**
```
DiceSymbol  →  sentence   (describe_scene)
DiceSymbol  →  image      (render_scene_image)
```
Both sentence and image are derived from the `DiceSymbol`

**Notebook order:**
1. Imports & Configuration
2. Data Structures (`DiceSymbol`, `Die`, `DiceScene`)
3. Random Scene Generator
4. Text Label Generator
5. Image Renderer
6. Dataset Generator
7. PyTorch Dataset & DataLoader

## 1. Imports & Configuration

In [ ]:
import os
print(os.getcwd())

/content


In [ ]:
import torch
print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())

False
False


In [ ]:
# ── Standard library ───────────────────────────────────────────────────────
import csv
import logging
import os
import random
import shutil
import time
import warnings
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

# ── Scientific / data ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Image processing ───────────────────────────────────────────────────────
from PIL import Image, ImageDraw

# ── PyTorch core ───────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ── Torchvision ────────────────────────────────────────────────────────────
from torchvision import models, transforms
from torchvision.transforms import v2

# ── Scikit-learn ───────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# ── NLP / embeddings ───────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

# ── Plotting ───────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Project ────────────────────────────────────────────────────────────────
from eval import evaluate_batch

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# Global config
CANVAS_SIZE = 320
MARGIN      = 20
MIN_DIE_SIZE = 60
MAX_DIE_SIZE = 100

# Discrete size categories and their pixel ranges
SIZE_CATEGORIES = {
    "small":  (60, 69),
    "medium": (70, 89),
    "large":  (90, 100),
}
VALID_SIZES = list(SIZE_CATEGORIES.keys())

COLOUR_MAP = {
    "white":  (255, 255, 255),
    "red":    (255, 220, 220),
    "blue":   (220, 235, 255),
    "green":  (220, 255, 220),
    "yellow": (255, 245, 200),
    "purple": (235, 220, 255),
    "peach":  (255, 230, 210),
}
VALID_COLOURS = list(COLOUR_MAP.keys())

DIE_OUTLINE = (40, 40, 40)
PIP_COLOR   = (20, 20, 20)

## 2. Data Structures

`DiceSymbol` is the canonical intermediate representation, a structured, discrete description of the scene. Both the sentence and the image are derived from it, never from each other directly.

In [ ]:
def discretise_size(px: int) -> str:
    """Map a pixel size to a discrete size category."""
    for label, (lo, hi) in SIZE_CATEGORIES.items():
        if lo <= px <= hi:
            return label
    raise ValueError(f"Pixel size {px} outside expected range {MIN_DIE_SIZE}-{MAX_DIE_SIZE}.")


def size_to_pixels(size_label: str) -> int:
    """Sample a random pixel size from within a size category."""
    if size_label not in SIZE_CATEGORIES:
        raise ValueError(f"Unknown size label '{size_label}'. Must be one of {VALID_SIZES}.")
    lo, hi = SIZE_CATEGORIES[size_label]
    return random.randint(lo, hi)


@dataclass
class DiceSymbol:
    """
    Canonical structured representation of a dice scene.
    This is the ground truth the neural network learns to predict.
    All fields are discrete symbols.
    """
    num_dice: int          # 1, 2, or 3
    values:   list         # e.g. [3, 5]
    colours:  list         # e.g. ["blue", "red"]
    sizes:    list         # e.g. ["large", "small"]

    def __post_init__(self):
        if not (1 <= self.num_dice <= 3):
            raise ValueError(f"num_dice must be 1-3, got {self.num_dice}.")
        if len(self.values) != self.num_dice:
            raise ValueError(f"Expected {self.num_dice} values, got {len(self.values)}.")
        if len(self.colours) != self.num_dice:
            raise ValueError(f"Expected {self.num_dice} colours, got {len(self.colours)}.")
        if len(self.sizes) != self.num_dice:
            raise ValueError(f"Expected {self.num_dice} sizes, got {len(self.sizes)}.")
        for v in self.values:
            if not (1 <= v <= 6):
                raise ValueError(f"Die value must be 1-6, got {v}.")
        for c in self.colours:
            if c not in VALID_COLOURS:
                raise ValueError(f"Invalid colour '{c}'. Must be one of {VALID_COLOURS}.")
        for s in self.sizes:
            if s not in VALID_SIZES:
                raise ValueError(f"Invalid size '{s}'. Must be one of {VALID_SIZES}.")

    def __str__(self):
        dice_strs = [
            f"Die(value={v}, colour={c}, size={s})"
            for v, c, s in zip(self.values, self.colours, self.sizes)
        ]
        return f"DiceSymbol({self.num_dice} dice: {' | '.join(dice_strs)})"


@dataclass
class Die:
    """Rendering-level die — extends DiceSymbol with pixel-level position/angle."""
    value:  int
    x:      int
    y:      int
    size:   int   = 80
    angle:  float = 0.0
    colour: str   = "white"

    def __post_init__(self):
        if self.colour not in COLOUR_MAP:
            raise ValueError(f"Invalid colour '{self.colour}'. Must be one of {VALID_COLOURS}.")
        if not (1 <= self.value <= 6):
            raise ValueError(f"Die value must be 1-6, got {self.value}.")

    @property
    def size_label(self) -> str:
        """Return the discrete size category for this die."""
        return discretise_size(self.size)


@dataclass
class DiceScene:
    dice: list = field(default_factory=list)

    def __post_init__(self):
        self._validate()

    def _validate(self):
        if not (1 <= len(self.dice) <= 3):
            raise ValueError(f"A scene must contain 1-3 dice, got {len(self.dice)}.")
        for die in self.dice:
            if not (MIN_DIE_SIZE <= die.size <= MAX_DIE_SIZE):
                raise ValueError(f"Die size must be {MIN_DIE_SIZE}-{MAX_DIE_SIZE}, got {die.size}.")
        positions = [(die.x, die.y) for die in self.dice]
        if len(positions) != len(set(positions)):
            raise ValueError("Two dice cannot have exactly the same top-left position.")

    def to_symbol(self) -> DiceSymbol:
        """Extract the canonical DiceSymbol from this scene."""
        return DiceSymbol(
            num_dice=len(self.dice),
            values=[d.value for d in self.dice],
            colours=[d.colour for d in self.dice],
            sizes=[d.size_label for d in self.dice],
        )

    def count(self) -> int:
        return len(self.dice)

    def values(self) -> list:
        return [die.value for die in self.dice]

    def __str__(self) -> str:
        return " | ".join(
            [f"Die(value={d.value}, colour={d.colour}, size={d.size_label}, x={d.x}, y={d.y})" for d in self.dice]
        )

## 3. Random Scene Generator

In [ ]:
def _overlap(d1: Die, d2: Die, padding: int = 10) -> bool:
    """Return True if two dice bounding boxes overlap (with padding)."""
    return not (
        d1.x + d1.size + padding <= d2.x or
        d2.x + d2.size + padding <= d1.x or
        d1.y + d1.size + padding <= d2.y or
        d2.y + d2.size + padding <= d1.y
    )


def _is_valid_placement(new_die: Die, placed_dice: list, padding: int = 10) -> bool:
    """Return True if new_die does not overlap any already-placed dice."""
    return all(not _overlap(new_die, die, padding=padding) for die in placed_dice)


def random_symbol(num_dice: int = None) -> DiceSymbol:
    """
    Generate a random DiceSymbol — the structured ground truth.
    This is the first stage of the pipeline.
    """
    if num_dice is None:
        num_dice = random.randint(1, 3)
    if not (1 <= num_dice <= 3):
        raise ValueError(f"num_dice must be 1-3, got {num_dice}.")

    return DiceSymbol(
        num_dice=num_dice,
        values=[random.randint(1, 6) for _ in range(num_dice)],
        colours=[random.choice(VALID_COLOURS) for _ in range(num_dice)],
        sizes=[random.choice(VALID_SIZES) for _ in range(num_dice)],
    )


def symbol_to_scene(
    symbol: DiceSymbol,
    canvas_size: int = CANVAS_SIZE,
    max_attempts: int = 200
) -> DiceScene:
    """
    Render a DiceSymbol into a DiceScene by assigning pixel-level
    positions and angles. This is the second stage of the pipeline.
    """
    dice = []

    for i in range(symbol.num_dice):
        placed = False

        for _ in range(max_attempts):
            size = size_to_pixels(symbol.sizes[i])
            max_x = canvas_size - size - MARGIN
            max_y = canvas_size - size - MARGIN

            if max_x < MARGIN or max_y < MARGIN:
                raise ValueError(
                    f"Canvas too small ({canvas_size}px) for die of size {size}px "
                    f"with margin {MARGIN}px."
                )

            x     = random.randint(MARGIN, max_x)
            y     = random.randint(MARGIN, max_y)
            angle = random.uniform(-15, 15)

            candidate = Die(
                value=symbol.values[i],
                x=x, y=y,
                size=size,
                angle=angle,
                colour=symbol.colours[i]
            )

            if _is_valid_placement(candidate, dice):
                dice.append(candidate)
                placed = True
                break

        if not placed:
            raise RuntimeError(
                f"Could not place die {i+1}/{symbol.num_dice} without overlap after "
                f"{max_attempts} attempts. Try a larger canvas or fewer dice."
            )

    return DiceScene(dice=dice)


def random_scene(num_dice: int = None) -> tuple:
    """
    Convenience wrapper: generate a random symbol and convert it to a scene.
    Returns (DiceSymbol, DiceScene) so both are always available.
    """
    symbol = random_symbol(num_dice)
    scene  = symbol_to_scene(symbol)
    return symbol, scene


## 4. Text Label Generator

Sentences are generated from `DiceSymbol` fields, including size. Multiple templates ensure variety.

In [ ]:
NUMBER_WORDS = {
    1: "one", 2: "two", 3: "three",
    4: "four", 5: "five", 6: "six",
}
COUNT_WORDS = {
    1: "one die", 2: "two dice", 3: "three dice",
}


def _value_word(v: int) -> str:
    if v not in NUMBER_WORDS:
        raise ValueError(f"Die value must be 1-6, got {v}.")
    return NUMBER_WORDS[v]


def _join_phrases(items: list) -> str:
    """Join a list of strings naturally: 'a, b and c'."""
    if not items:
        raise ValueError("Cannot join an empty list.")
    if len(items) == 1:
        return items[0]
    if len(items) == 2:
        return items[0] + " and " + items[1]
    return ", ".join(items[:-1]) + " and " + items[-1]


def describe_symbol(symbol: DiceSymbol) -> str:
    """
    Generate a natural language sentence from a DiceSymbol.
    Includes value, colour, AND size so all symbol fields appear in the text.
    """
    value_words = [_value_word(v) for v in symbol.values]
    count       = symbol.num_dice
    colours     = symbol.colours
    sizes       = symbol.sizes

    all_same_colour = len(set(colours)) == 1
    all_same_size   = len(set(sizes)) == 1

    template = random.randint(1, 4)

    if count == 1:
        s, c, v = sizes[0], colours[0], value_words[0]
        templates = [
            f"There is one {s} {c} die showing {v}.",
            f"The image shows one {s} {c} die with value {v}.",
            f"This picture contains a {s} {c} die displaying {v}.",
            f"A {s} {c} die is shown with a value of {v}.",
        ]

    elif all_same_colour and all_same_size:
        s, c = sizes[0], colours[0]
        vals = _join_phrases(value_words)
        templates = [
            f"There are {count} {s} {c} dice showing {vals}.",
            f"The image shows {count} {s} {c} dice with values {vals}.",
            f"This picture contains {count} {s} {c} dice displaying {vals}.",
            f"A scene with {count} {s} {c} dice is shown, displaying {vals}.",
        ]

    else:
        # Describe each die individually with its own size, colour, and value
        parts = [
            f"a {s} {c} die showing {v}"
            for s, c, v in zip(sizes, colours, value_words)
        ]
        joined = _join_phrases(parts)
        templates = [
            f"The image shows {joined}.",
            f"This picture contains {joined}.",
            f"There are {joined} in the scene.",
            f"The scene contains {joined}.",
        ]

    return templates[template - 1]


## 5. Image Renderer

Renders a `DiceScene` into a PIL image. Note: the renderer takes a `DiceScene` (which has pixel-level detail), not a `DiceSymbol` directly.

In [ ]:
PIP_MAP = {
    1: ["mc"],
    2: ["tl", "br"],
    3: ["tl", "mc", "br"],
    4: ["tl", "tr", "bl", "br"],
    5: ["tl", "tr", "mc", "bl", "br"],
    6: ["tl", "tr", "ml", "mr", "bl", "br"],
}


def _random_background() -> tuple:
    """Return a slightly varied light background colour."""
    return tuple(random.randint(235, 250) for _ in range(3))


def _pip_positions(x: int, y: int, size: int) -> dict:
    """Return standard pip anchor positions for a die face."""
    left  = x + size * 0.25
    cx    = x + size * 0.5
    right = x + size * 0.75
    top   = y + size * 0.25
    cy    = y + size * 0.5
    bot   = y + size * 0.75
    return {
        "tl": (left, top),  "tc": (cx, top),   "tr": (right, top),
        "ml": (left, cy),   "mc": (cx, cy),     "mr": (right, cy),
        "bl": (left, bot),  "bc": (cx, bot),    "br": (right, bot),
    }


def _draw_pip(draw: ImageDraw.ImageDraw, cx: float, cy: float, r: int = 6):
    draw.ellipse((cx - r, cy - r, cx + r, cy + r), fill=PIP_COLOR)


def _draw_die(img: Image.Image, die: Die):
    """Draw a single die onto the canvas image."""
    if die.value not in PIP_MAP:
        raise ValueError(f"Cannot draw die with value {die.value}. Must be 1-6.")
    if die.colour not in COLOUR_MAP:
        raise ValueError(f"Unknown colour '{die.colour}'.")

    size = die.size
    pad  = 20
    tile = size + pad * 2

    die_img = Image.new("RGBA", (tile, tile), (0, 0, 0, 0))
    draw    = ImageDraw.Draw(die_img)

    x0, y0 = pad, pad
    x1, y1 = pad + size, pad + size

    # Soft shadow
    so = 4
    draw.rounded_rectangle(
        (x0 + so, y0 + so, x1 + so, y1 + so),
        radius=12, fill=(0, 0, 0, 50)
    )
    # Die body
    draw.rounded_rectangle(
        (x0, y0, x1, y1),
        radius=12,
        fill=COLOUR_MAP[die.colour],
        outline=DIE_OUTLINE,
        width=3
    )
    # Pips
    pos   = _pip_positions(x0, y0, size)
    pip_r = max(4, size // 12)
    for key in PIP_MAP[die.value]:
        _draw_pip(draw, *pos[key], r=pip_r)

    rotated = die_img.rotate(die.angle, expand=True, resample=Image.Resampling.BICUBIC)
    paste_x = die.x - (rotated.width  - size) // 2
    paste_y = die.y - (rotated.height - size) // 2
    img.paste(rotated, (paste_x, paste_y), rotated)


def render_scene_image(
    scene: DiceScene,
    canvas_size: int = CANVAS_SIZE,
    background_color: tuple = None
) -> Image.Image:
    """Render a DiceScene into a PIL RGB image."""
    if background_color is None:
        background_color = _random_background()
    if len(background_color) != 3 or not all(0 <= c <= 255 for c in background_color):
        raise ValueError(f"background_color must be an RGB tuple with values 0-255.")

    img = Image.new("RGB", (canvas_size, canvas_size), background_color)
    for die in scene.dice:
        _draw_die(img, die)
    return img

## 6. Dataset Generator

Generates `total_samples` examples. Each sample stores:
- The rendered image (PNG)
- The natural language sentence
- The explicit symbol fields (`sym_num_dice`, `sym_values`, `sym_colours`, `sym_sizes`)

The symbol fields are the ground truth for the neural network.

In [ ]:
def make_clean_dir(path: str):
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)


def split_counts(total: int, train_ratio: float = 0.7, val_ratio: float = 0.15, test_ratio: float = 0.15):
    if not abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6:
        raise ValueError("Split ratios must sum to 1.0.")
    train_n = int(total * train_ratio)
    val_n   = int(total * val_ratio)
    test_n  = total - train_n - val_n
    return train_n, val_n, test_n


def generate_dataset(output_dir="dataset", total_samples=5000, seed=42, max_attempts_per_sample=20):
    if total_samples < 3:
        raise ValueError("total_samples must be at least 3.")

    random.seed(seed)
    np.random.seed(seed)

    make_clean_dir(output_dir)
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(output_dir, split, "images"), exist_ok=True)

    train_n, val_n, test_n = split_counts(total_samples)
    split_plan = ["train"] * train_n + ["val"] * val_n + ["test"] * test_n
    random.shuffle(split_plan)

    split_rows = {"train": [], "val": [], "test": []}
    successful = 0
    attempts   = 0

    while successful < total_samples:
        attempts += 1
        if attempts > total_samples * max_attempts_per_sample:
            raise RuntimeError(
                f"Gave up after {attempts} attempts — only {successful}/{total_samples} "
                f"samples generated. Check your renderer for systematic errors."
            )

        try:
            symbol   = random_symbol()
            sentence = describe_symbol(symbol)
            scene    = symbol_to_scene(symbol)
            image    = render_scene_image(scene)
        except Exception as e:
            logger.warning(f"Attempt {attempts}: generation failed — {e}")
            continue

        split          = split_plan[successful]
        image_filename = f"sample_{successful:05d}.png"
        image_path     = os.path.join(output_dir, split, "images", image_filename)

        try:
            image.save(image_path)
        except Exception as e:
            logger.warning(f"Attempt {attempts}: save failed — {e}")
            continue

        # Verify the file was written correctly before recording it
        try:
            with Image.open(image_path) as img:
                img.verify()
        except Exception as e:
            logger.warning(f"Attempt {attempts}: saved file is corrupt — {e}")
            os.remove(image_path)
            continue

        split_rows[split].append({
            "image":        image_filename,
            "text":         sentence,
            "sym_num_dice": symbol.num_dice,
            "sym_values":   " ".join(str(v) for v in symbol.values),
            "sym_colours":  " ".join(symbol.colours),
            "sym_sizes":    " ".join(symbol.sizes),
        })

        successful += 1
        if successful % 1000 == 0:
            logger.info(f"Generated {successful}/{total_samples} samples ({attempts} attempts)...")

    fieldnames = ["image", "text", "sym_num_dice", "sym_values", "sym_colours", "sym_sizes"]
    for split in ["train", "val", "test"]:
        csv_path = os.path.join(output_dir, split, "labels.csv")
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(split_rows[split])

    meta_path = os.path.join(output_dir, "dataset_info.txt")
    with open(meta_path, "w", encoding="utf-8") as f:
        f.write(f"total_samples: {total_samples}\n")
        f.write(f"successfully_generated: {successful}\n")
        f.write(f"attempts: {attempts}\n")
        f.write(f"seed: {seed}\n")
        f.write(f"train_count: {len(split_rows['train'])}\n")
        f.write(f"val_count: {len(split_rows['val'])}\n")
        f.write(f"test_count: {len(split_rows['test'])}\n")
        f.write("split_ratio: 70/15/15\n")
        f.write(f"symbol_fields: sym_num_dice, sym_values, sym_colours, sym_sizes\n")
        f.write(f"size_categories: {SIZE_CATEGORIES}\n")

    print(f"\nDataset generated successfully ({attempts} attempts for {total_samples} samples).")
    print(f"  Train: {len(split_rows['train'])}, Val: {len(split_rows['val'])}, Test: {len(split_rows['test'])}")


generate_dataset(output_dir="dataset", total_samples=6000, seed=42)


Dataset generated successfully (6022 attempts for 6000 samples).
  Train: 4200, Val: 900, Test: 900


## 7. PyTorch Dataset & DataLoader

The dataset returns both the text and the explicit symbol tensors. The symbol tensors (`sym_num_dice`, `sym_values`, `sym_colours`, `sym_sizes`) are the direct prediction targets for the neural network.

In [ ]:
# Index maps for encoding symbol fields as class indices
COLOUR_TO_IDX = {c: i for i, c in enumerate(VALID_COLOURS)}
SIZE_TO_IDX   = {s: i for i, s in enumerate(VALID_SIZES)}

print("Colour index map:", COLOUR_TO_IDX)
print("Size index map:  ", SIZE_TO_IDX)


class DiceDataset(Dataset):
    """
    PyTorch Dataset for the synthetic dice dataset.

    Each item returns:
        image       — float32 tensor [3, H, W]
        label dict  — containing text and symbol tensors

    Symbol tensors (ground truth for the neural network):
        sym_num_dice  — scalar long tensor (1-3)
        sym_values    — long tensor [3], padded with 0
        sym_colours   — long tensor [3], padded with -1
        sym_sizes     — long tensor [3], padded with -1
    """

    VALID_SPLITS = ("train", "val", "test")

    def __init__(self, root_dir: str, split: str = "train", transform=None):
        if split not in self.VALID_SPLITS:
            raise ValueError(f"split must be one of {self.VALID_SPLITS}, got '{split}'.")

        self.root_dir  = root_dir
        self.split     = split
        self.image_dir = os.path.join(root_dir, split, "images")
        self.csv_path  = os.path.join(root_dir, split, "labels.csv")

        if not os.path.isdir(self.image_dir):
            raise FileNotFoundError(
                f"Image directory not found: '{self.image_dir}'.\n"
                "Have you run generate_dataset() first?"
            )
        if not os.path.isfile(self.csv_path):
            raise FileNotFoundError(
                f"Labels file not found: '{self.csv_path}'.\n"
                "Have you run generate_dataset() first?"
            )

        self.samples = []
        with open(self.csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.samples.append(row)

        if len(self.samples) == 0:
            raise RuntimeError(f"No samples found in {self.csv_path}.")

        self.transform = transform or v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
        ])

        logger.info(f"DiceDataset ({split}): {len(self.samples)} samples loaded.")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        if not (0 <= idx < len(self.samples)):
            raise IndexError(f"Index {idx} out of range for dataset of size {len(self.samples)}.")

        row      = self.samples[idx]
        img_path = os.path.join(self.image_dir, row["image"])

        if not os.path.isfile(img_path):
            raise FileNotFoundError(f"Image file missing: '{img_path}'.")

        try:
            image = Image.open(img_path).convert("RGB")
            image = self.transform(image)
        except Exception as e:
            raise RuntimeError(f"Failed to load/transform image at '{img_path}': {e}") from e

        # Parse symbol fields
        try:
            num_dice = int(row["sym_num_dice"])
            values   = list(map(int,   row["sym_values"].split()))
            colours  = row["sym_colours"].split()
            sizes    = row["sym_sizes"].split()
        except (ValueError, KeyError) as e:
            raise ValueError(f"Malformed symbol fields in row {idx}: {e}") from e

        # Validate
        if not (1 <= num_dice <= 3):
            raise ValueError(f"sym_num_dice out of range in row {idx}: {num_dice}")
        if not all(1 <= v <= 6 for v in values):
            raise ValueError(f"sym_values out of range in row {idx}: {values}")
        if not all(c in COLOUR_TO_IDX for c in colours):
            raise ValueError(f"Unknown colour in row {idx}: {colours}")
        if not all(s in SIZE_TO_IDX for s in sizes):
            raise ValueError(f"Unknown size in row {idx}: {sizes}")

        # Pad to length 3 (max dice)
        padded_values  = values  + [0]  * (3 - len(values))
        padded_colours = [COLOUR_TO_IDX[c] for c in colours] + [-1] * (3 - len(colours))
        padded_sizes   = [SIZE_TO_IDX[s]   for s in sizes]   + [-1] * (3 - len(sizes))

        label = {
            "text":         row["text"],
            "sym_num_dice": torch.tensor(num_dice,       dtype=torch.long),
            "sym_values":   torch.tensor(padded_values,  dtype=torch.long),
            "sym_colours":  torch.tensor(padded_colours, dtype=torch.long),
            "sym_sizes":    torch.tensor(padded_sizes,   dtype=torch.long),
        }

        return image, label

Colour index map: {'white': 0, 'red': 1, 'blue': 2, 'green': 3, 'yellow': 4, 'purple': 5, 'peach': 6}
Size index map:   {'small': 0, 'medium': 1, 'large': 2}


In [ ]:
def dice_collate_fn(batch):
    """Custom collate to handle variable-length string 'text' field."""
    images = torch.stack([item[0] for item in batch])
    labels = {
        "text":         [item[1]["text"]         for item in batch],
        "sym_num_dice": torch.stack([item[1]["sym_num_dice"] for item in batch]),
        "sym_values":   torch.stack([item[1]["sym_values"]   for item in batch]),
        "sym_colours":  torch.stack([item[1]["sym_colours"]  for item in batch]),
        "sym_sizes":    torch.stack([item[1]["sym_sizes"]     for item in batch]),
    }
    return images, labels

N_WORKERS = min(4, os.cpu_count())

try:
    train_dataset = DiceDataset("dataset", split="train")
    val_dataset   = DiceDataset("dataset", split="val")
    test_dataset  = DiceDataset("dataset", split="test")

    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,  num_workers=N_WORKERS, collate_fn=dice_collate_fn)
    val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False, num_workers=N_WORKERS, collate_fn=dice_collate_fn)
    test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=N_WORKERS, collate_fn=dice_collate_fn)

    print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

    for imgs, labels in train_loader:
        print("Image shape:        ", imgs.shape)                    # [32, 3, 320, 320]
        print("sym_num_dice shape: ", labels["sym_num_dice"].shape)  # [32]
        print("sym_values shape:   ", labels["sym_values"].shape)    # [32, 3]
        print("sym_colours shape:  ", labels["sym_colours"].shape)   # [32, 3]
        print("sym_sizes shape:    ", labels["sym_sizes"].shape)     # [32, 3]
        print("Example text:       ", labels["text"][0])
        print("Example num_dice:   ", labels["sym_num_dice"][0].item())
        print("Example values:     ", labels["sym_values"][0].tolist())
        print("Example colours:    ", labels["sym_colours"][0].tolist())
        print("Example sizes:      ", labels["sym_sizes"][0].tolist())
        break

except FileNotFoundError as e:
    print(f"Dataset not found. Run generate_dataset() first.\nDetails: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")
    raise

Train batches: 17 | Val: 4 | Test: 4
Image shape:         torch.Size([256, 3, 320, 320])
sym_num_dice shape:  torch.Size([256])
sym_values shape:    torch.Size([256, 3])
sym_colours shape:   torch.Size([256, 3])
sym_sizes shape:     torch.Size([256, 3])
Example text:        This picture contains a small red die showing three and a large yellow die showing three.
Example num_dice:    2
Example values:      [3, 3, 0]
Example colours:     [1, 4, -1]
Example sizes:       [0, 2, -1]


# Axis 1 — Text Representation Comparison

Fixed backbone (SimpleCNN), three output spaces: One-Hot · TF-IDF · SBERT.

**Hypothesis:** SBERT embeddings will generalise better to unseen dice combinations
because they encode semantic similarity, whereas one-hot treats each sentence as
independent and TF-IDF captures only lexical overlap.

**Notebook order:**
1. Imports & Configuration
2. Load Data
3. Symbol Field Overview
4. One-Hot Encoding
5. TF-IDF Encoding
6. SBERT Encoding
7. Prepare Targets
8. Image Dataset & DataLoaders
9. Backbone & Prediction Heads
10. Train — One-Hot
11. Train — TF-IDF
12. Train — SBERT
13. Results & Evaluation
14. Loss Curves
15. Test Set Evaluation
16. Final Results Table

## 1. Imports & Configuration

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# Must match the generator notebook
DATASET_ROOT = "dataset"
SPLITS       = ["train", "val", "test"]

LABEL_PATHS = {
    split: os.path.join(DATASET_ROOT, split, "labels.csv")
    for split in SPLITS
}

# Symbol field config — must match generator notebook
VALID_COLOURS = ["white", "red", "blue", "green", "yellow", "purple", "peach"]
VALID_SIZES   = ["small", "medium", "large"]

# Verify files exist before proceeding
missing = [p for p in LABEL_PATHS.values() if not os.path.isfile(p)]
if missing:
    raise FileNotFoundError(
        f"Missing label files: {missing}\n"
        "Please run the dataset generator notebook first."
    )

print("All label files found. Ready to proceed.")

All label files found. Ready to proceed.


## 2. Load Data

In [ ]:
REQUIRED_COLUMNS = {"text", "sym_num_dice", "sym_values", "sym_colours", "sym_sizes", "image"}


def load_split(csv_path: str, split_name: str) -> pd.DataFrame:
    """Load a split CSV and validate expected columns and content."""
    df = pd.read_csv(csv_path)

    missing_cols = REQUIRED_COLUMNS - set(df.columns)
    if missing_cols:
        raise ValueError(
            f"[{split_name}] CSV missing columns: {missing_cols}.\n"
            "Ensure you ran the latest version of the generator notebook."
        )
    if df.empty:
        raise ValueError(f"[{split_name}] CSV is empty.")
    if df["text"].isnull().any():
        raise ValueError(f"[{split_name}] Found null values in 'text' column.")

    logger.info(f"Loaded {split_name}: {len(df)} samples.")
    return df


train_df = load_split(LABEL_PATHS["train"], "train")
val_df   = load_split(LABEL_PATHS["val"],   "val")
test_df  = load_split(LABEL_PATHS["test"],  "test")

train_texts = train_df["text"].tolist()
val_texts   = val_df["text"].tolist()
test_texts  = test_df["text"].tolist()

print(f"\nSplit sizes — Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")
print(f"\nSymbol columns present: {[c for c in train_df.columns if c.startswith('sym_')]}")
print(f"\nExample row:")
train_df.head(3)


Split sizes — Train: 4200, Val: 900, Test: 900

Symbol columns present: ['sym_num_dice', 'sym_values', 'sym_colours', 'sym_sizes']

Example row:


,image,text,sym_num_dice,sym_values,sym_colours,sym_sizes
0,sample_00000.png,The image shows a small purple die showing two...,3,2 1 2,purple green blue,small small medium
1,sample_00004.png,The scene contains a small peach die showing f...,3,4 3 5,peach yellow red,small large small
2,sample_00007.png,This picture contains a medium green die showi...,3,5 3 6,green purple purple,medium small small


## 3. Symbol Field Overview

Quick look at the distribution of the ground truth symbol fields.

In [ ]:
print("=== Symbol field distributions (train set) ===")
print("\nnum_dice distribution:")
print(train_df["sym_num_dice"].value_counts().sort_index())

# Flatten space-separated colour and size fields to get per-die distributions
all_colours = [c for row in train_df["sym_colours"] for c in row.split()]
all_sizes   = [s for row in train_df["sym_sizes"]   for s in row.split()]
all_values  = [int(v) for row in train_df["sym_values"] for v in row.split()]

print("\nColour distribution (per die):")
colour_series = pd.Series(all_colours).value_counts()
print(colour_series)

print("\nSize distribution (per die):")
size_series = pd.Series(all_sizes).value_counts().reindex(VALID_SIZES)
print(size_series)

print("\nFace value distribution (per die):")
value_series = pd.Series(all_values).value_counts().sort_index()
print(value_series)

# Validate no unexpected values
unexpected_colours = set(all_colours) - set(VALID_COLOURS)
unexpected_sizes   = set(all_sizes)   - set(VALID_SIZES)
assert not unexpected_colours, f"Unexpected colours found: {unexpected_colours}"
assert not unexpected_sizes,   f"Unexpected sizes found: {unexpected_sizes}"
print("\nValidation passed: all symbol values are within expected ranges.")

=== Symbol field distributions (train set) ===

num_dice distribution:
sym_num_dice
1    1416
2    1411
3    1373
Name: count, dtype: int64

Colour distribution (per die):
yellow    1222
green     1215
red       1210
peach     1207
blue      1195
purple    1156
white     1152
Name: count, dtype: int64

Size distribution (per die):
small     2784
medium    2861
large     2712
Name: count, dtype: int64

Face value distribution (per die):
1    1424
2    1377
3    1392
4    1395
5    1370
6    1399
Name: count, dtype: int64

Validation passed: all symbol values are within expected ranges.


## 4. One-Hot Encoding

Treats each unique sentence as a category. Included as a weak baseline only — the high UNK rate on val/test makes it unsuitable for generalisation.

> With the richer sentences (now including size), expect more unique sentences and a higher UNK rate than before.

In [ ]:
def build_sentence_vocab(texts: list) -> tuple:
    """Build a sentence-to-id mapping from training texts only."""
    unique_texts  = sorted(set(texts))
    sentence_to_id = {text: i for i, text in enumerate(unique_texts)}
    unk_id = len(sentence_to_id)
    return sentence_to_id, unk_id


def one_hot_encode(text: str, sentence_to_id: dict, unk_id: int) -> np.ndarray:
    vocab_size = len(sentence_to_id) + 1
    vec = np.zeros(vocab_size, dtype=np.float32)
    vec[sentence_to_id.get(text, unk_id)] = 1.0
    return vec


def encode_split_onehot(texts: list, sentence_to_id: dict, unk_id: int) -> np.ndarray:
    return np.array([one_hot_encode(t, sentence_to_id, unk_id) for t in texts])


sentence_to_id, UNK_ID = build_sentence_vocab(train_texts)
logger.info(f"One-hot vocab: {len(sentence_to_id)} unique training sentences + 1 UNK.")

X_train_onehot = encode_split_onehot(train_texts, sentence_to_id, UNK_ID)
X_val_onehot   = encode_split_onehot(val_texts,   sentence_to_id, UNK_ID)
X_test_onehot  = encode_split_onehot(test_texts,  sentence_to_id, UNK_ID)

print("Train one-hot shape:", X_train_onehot.shape)
print("Val one-hot shape:  ", X_val_onehot.shape)
print("Test one-hot shape: ", X_test_onehot.shape)

assert np.all(X_train_onehot.sum(axis=1) == 1.0)
assert np.all(X_val_onehot.sum(axis=1)   == 1.0)
assert np.all(X_test_onehot.sum(axis=1)  == 1.0)
print("Sanity check passed: all rows sum to 1.0")

unk_val  = int(np.sum(np.argmax(X_val_onehot,  axis=1) == UNK_ID))
unk_test = int(np.sum(np.argmax(X_test_onehot, axis=1) == UNK_ID))
print(f"\nUNK in val:  {unk_val}/{len(val_texts)}   ({100*unk_val/len(val_texts):.1f}% unseen)")
print(f"UNK in test: {unk_test}/{len(test_texts)} ({100*unk_test/len(test_texts):.1f}% unseen)")
print(f"\nExample: '{train_texts[0]}'")
print(f"One-hot: {X_train_onehot[0]}")

Train one-hot shape: (4200, 3238)
Val one-hot shape:   (900, 3238)
Test one-hot shape:  (900, 3238)
Sanity check passed: all rows sum to 1.0

UNK in val:  622/900   (69.1% unseen)
UNK in test: 619/900 (68.8% unseen)

Example: 'The image shows a small purple die showing two, a small green die showing one and a medium blue die showing two.'
One-hot: [0. 0. 0. ... 0. 0. 0.]


## 5. TF-IDF

Word + bigram TF-IDF, fitted on training data only. Now that sentences include size words (small / medium / large), the vocabulary and feature space will be richer than before.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),  # unigrams + bigrams (captures e.g. 'large blue', 'showing three')
    min_df=2,            # drop terms appearing in fewer than 2 documents
    sublinear_tf=True,   # log(1 + tf) dampens high raw frequencies
)

# Fit on training data only
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_val_tfidf   = tfidf_vectorizer.transform(val_texts)
X_test_tfidf  = tfidf_vectorizer.transform(test_texts)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Val TF-IDF shape:  ", X_val_tfidf.shape)
print("Test TF-IDF shape: ", X_test_tfidf.shape)
print(f"Vocabulary size:    {len(tfidf_vectorizer.vocabulary_)} terms")

assert X_train_tfidf.shape[1] == X_val_tfidf.shape[1] == X_test_tfidf.shape[1]
print("Sanity check passed: feature dims match across splits.")

# Show top features for first example, sorted by weight
feature_names   = tfidf_vectorizer.get_feature_names_out()
example_vector  = X_train_tfidf[0].toarray()[0]
nonzero_indices = example_vector.nonzero()[0]

print(f"\nExample: '{train_texts[0]}'")
print("Non-zero TF-IDF values (sorted by weight):")
for idx in sorted(nonzero_indices, key=lambda i: -example_vector[i]):
    print(f"  {feature_names[idx]:<25} -> {example_vector[idx]:.4f}")

Train TF-IDF shape: (4200, 178)
Val TF-IDF shape:   (900, 178)
Test TF-IDF shape:  (900, 178)
Vocabulary size:    178 terms
Sanity check passed: feature dims match across splits.

Example: 'The image shows a small purple die showing two, a small green die showing one and a medium blue die showing two.'
Non-zero TF-IDF values (sorted by weight):
  two small                 -> 0.3408
  showing two               -> 0.2742
  shows small               -> 0.2590
  two                       -> 0.2539
  small purple              -> 0.2335
  medium blue               -> 0.2277
  small green               -> 0.2247
  one and                   -> 0.2097
  small                     -> 0.1855
  die showing               -> 0.1845
  showing                   -> 0.1838
  and medium                -> 0.1685
  image                     -> 0.1613
  image shows               -> 0.1613
  shows                     -> 0.1613
  the image                 -> 0.1613
  purple die                -> 0.1600
  purpl

## 6. Sentence-BERT (SBERT)

Dense 384-dimensional semantic embeddings. Handles all unseen sentences and captures meaning — semantically similar sentences (e.g. same dice, different template) will have similar vectors.

In [ ]:
try:
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
except Exception as e:
    raise RuntimeError(
        f"Failed to load SentenceTransformer: {e}\n"
        "Install with: pip install sentence-transformers"
    ) from e

X_train_sbert = sbert_model.encode(train_texts, convert_to_numpy=True, show_progress_bar=True, batch_size=256)
X_val_sbert   = sbert_model.encode(val_texts,   convert_to_numpy=True, show_progress_bar=True, batch_size=256)
X_test_sbert  = sbert_model.encode(test_texts,  convert_to_numpy=True, show_progress_bar=True, batch_size=256)

print("Train SBERT shape:", X_train_sbert.shape)
print("Val SBERT shape:  ", X_val_sbert.shape)
print("Test SBERT shape: ", X_test_sbert.shape)

assert X_train_sbert.shape[1] == 384
assert X_train_sbert.shape[0] == len(train_texts)
assert not np.isnan(X_train_sbert).any(), "NaN values in SBERT embeddings"
print("\nSanity checks passed.")

# Quick semantic similarity check — sentences about the same symbol should be close
from numpy.linalg import norm
def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

sim_same  = cosine_sim(X_train_sbert[0], X_train_sbert[1])
sim_diff  = cosine_sim(X_train_sbert[0], X_train_sbert[-1])
print(f"\nExample similarity (consecutive sentences): {sim_same:.3f}")
print(f"Example similarity (distant sentences):     {sim_diff:.3f}")
print(f"\nExample: '{train_texts[0]}'")
print(f"SBERT (first 10): {X_train_sbert[0][:10]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Train SBERT shape: (4200, 384)
Val SBERT shape:   (900, 384)
Test SBERT shape:  (900, 384)

Sanity checks passed.

Example similarity (consecutive sentences): 0.610
Example similarity (distant sentences):     0.849

Example: 'The image shows a small purple die showing two, a small green die showing one and a medium blue die showing two.'
SBERT (first 10): [ 0.01368802 -0.02950342 -0.02164594 -0.02630392 -0.01849686 -0.00938264
  0.05967037 -0.01503856  0.0594265   0.01909663]


## 7. Prepare Targets for Axis 1

In [ ]:
#One-hot
Y_train_onehot = np.asarray(X_train_onehot, dtype=np.float32)
Y_val_onehot   = np.asarray(X_val_onehot, dtype=np.float32)
Y_test_onehot  = np.asarray(X_test_onehot, dtype=np.float32)

#TF-IDF
Y_train_tfidf = X_train_tfidf.toarray().astype(np.float32)
Y_val_tfidf   = X_val_tfidf.toarray().astype(np.float32)
Y_test_tfidf  = X_test_tfidf.toarray().astype(np.float32)

#SBERT
Y_train_sbert = np.asarray(X_train_sbert, dtype=np.float32)
Y_val_sbert   = np.asarray(X_val_sbert, dtype=np.float32)
Y_test_sbert  = np.asarray(X_test_sbert, dtype=np.float32)

#Collect them in dictionaries
axis1_targets = {
    "onehot": {
        "train": Y_train_onehot,
        "val":   Y_val_onehot,
        "test":  Y_test_onehot,
    },
    "tfidf": {
        "train": Y_train_tfidf,
        "val":   Y_val_tfidf,
        "test":  Y_test_tfidf,
    },
    "sbert": {
        "train": Y_train_sbert,
        "val":   Y_val_sbert,
        "test":  Y_test_sbert,
    },
}

target_dims = {
    "onehot": Y_train_onehot.shape[1],
    "tfidf":  Y_train_tfidf.shape[1],
    "sbert":  Y_train_sbert.shape[1],
}

#Sanity checks
assert Y_train_onehot.shape[0] == len(train_df)
assert Y_val_onehot.shape[0]   == len(val_df)
assert Y_test_onehot.shape[0]  == len(test_df)

assert Y_train_tfidf.shape[0] == len(train_df)
assert Y_val_tfidf.shape[0]   == len(val_df)
assert Y_test_tfidf.shape[0]  == len(test_df)

assert Y_train_sbert.shape[0] == len(train_df)
assert Y_val_sbert.shape[0]   == len(val_df)
assert Y_test_sbert.shape[0]  == len(test_df)

print("Axis 1 target preparation complete.\n")

print("One-hot shapes:")
print("  train:", Y_train_onehot.shape)
print("  val:  ", Y_val_onehot.shape)
print("  test: ", Y_test_onehot.shape)

print("\nTF-IDF shapes:")
print("  train:", Y_train_tfidf.shape)
print("  val:  ", Y_val_tfidf.shape)
print("  test: ", Y_test_tfidf.shape)

print("\nSBERT shapes:")
print("  train:", Y_train_sbert.shape)
print("  val:  ", Y_val_sbert.shape)
print("  test: ", Y_test_sbert.shape)

print("\nTarget dimensions:")
print(target_dims)

Axis 1 target preparation complete.

One-hot shapes:
  train: (4200, 3238)
  val:   (900, 3238)
  test:  (900, 3238)

TF-IDF shapes:
  train: (4200, 178)
  val:   (900, 178)
  test:  (900, 178)

SBERT shapes:
  train: (4200, 384)
  val:   (900, 384)
  test:  (900, 384)

Target dimensions:
{'onehot': 3238, 'tfidf': 178, 'sbert': 384}


## 9. Build Image Dataset and DataLoaders

In [ ]:
DATASET_ROOT = "dataset"

# A simple image transform
image_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

def resolve_image_path(dataset_root, split_name, image_name):
    """
    Try a few common locations for an image file.
    """
    candidates = [
        os.path.join(dataset_root, split_name, "images", image_name),
        os.path.join(dataset_root, split_name, image_name),
        image_name
    ]
    for path in candidates:
        if os.path.isfile(path):
            return path
    return None   # return None instead of raising — filtered below


def filter_valid_rows(df, split_name, dataset_root="dataset"):
    valid_indices = []
    for i, (_, row) in enumerate(df.iterrows()):
        path = resolve_image_path(dataset_root, split_name, row["image"])
        if path is None or not os.path.isfile(path) or os.path.getsize(path) == 0:
            continue
        try:
            with Image.open(path) as img:
                img.verify()
            valid_indices.append(i)
        except Exception:
            print(f"  [{split_name}] Skipping corrupt image: {path}")

    n_dropped = len(df) - len(valid_indices)
    if n_dropped > 0:
        print(f"  [{split_name}] Dropped {n_dropped} rows with missing/corrupt images.")

    cleaned = df.iloc[valid_indices].reset_index(drop=True)
    return cleaned, valid_indices


class Axis1ImageTextDataset(Dataset):
    def __init__(self, df, targets, split_name, dataset_root="dataset", transform=None):
        self.df = df.reset_index(drop=True)
        self.targets = targets
        self.split_name = split_name
        self.dataset_root = dataset_root
        self.transform = transform
        assert len(self.df) == len(self.targets), "df and targets must have same length"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_name = row["image"]
        image_path = resolve_image_path(self.dataset_root, self.split_name, image_name)

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception:
            # Corrupt or missing file — return a blank image so the batch completes
            logger.warning(f"Skipping corrupt image at idx {idx}: {image_name}")
            image = Image.new("RGB", (128, 128), color=(0, 0, 0))

        if self.transform is not None:
            image = self.transform(image)

        target = torch.tensor(self.targets[idx], dtype=torch.float32)

        return {
            "image": image,
            "target": target,
            "text": row["text"],
            "image_name": image_name,
        }


def make_dataloaders(target_name, batch_size=256):
    train_df_clean, train_idx = filter_valid_rows(train_df, "train", DATASET_ROOT)
    val_df_clean,   val_idx   = filter_valid_rows(val_df,   "val",   DATASET_ROOT)
    test_df_clean,  test_idx  = filter_valid_rows(test_df,  "test",  DATASET_ROOT)

    # Filter targets by the same indices, not a blind slice
    train_targets = [axis1_targets[target_name]["train"][i] for i in train_idx]
    val_targets   = [axis1_targets[target_name]["val"][i]   for i in val_idx]
    test_targets  = [axis1_targets[target_name]["test"][i]  for i in test_idx]

    train_dataset = Axis1ImageTextDataset(
        df=train_df_clean, targets=train_targets,
        split_name="train", dataset_root=DATASET_ROOT, transform=image_transform
    )
    val_dataset = Axis1ImageTextDataset(
        df=val_df_clean, targets=val_targets,
        split_name="val", dataset_root=DATASET_ROOT, transform=image_transform
    )
    test_dataset = Axis1ImageTextDataset(
        df=test_df_clean, targets=test_targets,
        split_name="test", dataset_root=DATASET_ROOT, transform=image_transform
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [ ]:
# Sanity check with one-hot targets
train_loader_onehot, val_loader_onehot, test_loader_onehot = make_dataloaders("onehot", batch_size=256)

batch = next(iter(train_loader_onehot))

print("Image batch shape:", batch["image"].shape)
print("Target batch shape:", batch["target"].shape)
print("First image name:", batch["image_name"][0])
print("First text:", batch["text"][0])

Image batch shape: torch.Size([256, 3, 128, 128])
Target batch shape: torch.Size([256, 3238])
First image name: sample_00542.png
First text: The image shows a medium peach die showing one and a large purple die showing six.


## 10. Choose a Fixed Image Backbone

In [ ]:
class SimpleCNNBackbone(nn.Module):
    def __init__(self, feature_dim=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # [B, 32, 320, 320]
            nn.ReLU(),
            nn.MaxPool2d(2),                              # [B, 32, 160, 160]

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # [B, 64, 160, 160]
            nn.ReLU(),
            nn.MaxPool2d(2),                              # [B, 64, 80, 80]

            nn.Conv2d(64, 128, kernel_size=3, padding=1),# [B, 128, 80, 80]
            nn.ReLU(),
            nn.MaxPool2d(2),                              # [B, 128, 40, 40]

            nn.Conv2d(128, 256, kernel_size=3, padding=1), # [B, 256, 40, 40]
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))                 # [B, 256, 1, 1]
        )

        self.feature_layer = nn.Linear(256, feature_dim)

    def forward(self, x):
        x = self.conv(x)                  # [B, 256, 1, 1]
        x = x.view(x.size(0), -1)         # [B, 256]
        x = self.feature_layer(x)         # [B, feature_dim]
        return x

FEATURE_DIM = 256
backbone = SimpleCNNBackbone(feature_dim=FEATURE_DIM)

print(backbone)
print("Feature dim:", FEATURE_DIM)

SimpleCNNBackbone(
  (conv): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (feature_layer): Linear(in_features=256, out_features=256, bias=True)
)
Feature dim: 256


In [ ]:
# Sanity check backbone output

sample_images = batch["image"]
features = backbone(sample_images)

print("Input image batch shape:", sample_images.shape)
print("Backbone output shape:", features.shape)

Input image batch shape: torch.Size([256, 3, 128, 128])
Backbone output shape: torch.Size([256, 256])


## 11. Define Prediction Heads

In [ ]:
class PredictionHead(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)

# Build one head for each text space
head_onehot = PredictionHead(input_dim=FEATURE_DIM, output_dim=target_dims["onehot"])
head_tfidf  = PredictionHead(input_dim=FEATURE_DIM, output_dim=target_dims["tfidf"])
head_sbert  = PredictionHead(input_dim=FEATURE_DIM, output_dim=target_dims["sbert"])

print("One-hot head output dim:", target_dims["onehot"])
print("TF-IDF head output dim:", target_dims["tfidf"])
print("SBERT head output dim:", target_dims["sbert"])

One-hot head output dim: 3238
TF-IDF head output dim: 178
SBERT head output dim: 384


In [ ]:
# Sanity check prediction heads

with torch.no_grad():
    sample_features = backbone(batch["image"])

    out_onehot = head_onehot(sample_features)
    out_tfidf  = head_tfidf(sample_features)
    out_sbert  = head_sbert(sample_features)

print("Feature shape:", sample_features.shape)
print("One-hot head output shape:", out_onehot.shape)
print("TF-IDF head output shape:", out_tfidf.shape)
print("SBERT head output shape:", out_sbert.shape)

Feature shape: torch.Size([256, 256])
One-hot head output shape: torch.Size([256, 3238])
TF-IDF head output shape: torch.Size([256, 178])
SBERT head output shape: torch.Size([256, 384])


## 12. Train the One-Hot Version

In [ ]:
class Axis1Model(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x):
        features = self.backbone(x)
        outputs = self.head(features)
        return outputs

model_onehot = Axis1Model(backbone=SimpleCNNBackbone(feature_dim=FEATURE_DIM),
                          head=PredictionHead(input_dim=FEATURE_DIM, output_dim=target_dims["onehot"]))

print(model_onehot)

Axis1Model(
  (backbone): SimpleCNNBackbone(
    (conv): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): ReLU()
      (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (10): ReLU()
      (11): AdaptiveAvgPool2d(output_size=(1, 1))
    )
    (feature_layer): Linear(in_features=256, out_features=256, bias=True)
  )
  (head): PredictionHead(
    (net): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_feat

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

model_onehot = model_onehot.to(device)

criterion_onehot = nn.CrossEntropyLoss()
optimizer_onehot = torch.optim.Adam(model_onehot.parameters(), lr=1e-3)

Using device: cuda


In [ ]:
def train_one_epoch_onehot(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in dataloader:
        images = batch["image"].to(device)
        targets_onehot = batch["target"].to(device)

        # convert one-hot vectors to class indices
        target_ids = torch.argmax(targets_onehot, dim=1)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, target_ids)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == target_ids).sum().item()
        total_samples += images.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc


@torch.no_grad()
def evaluate_onehot(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in dataloader:
        images = batch["image"].to(device)
        targets_onehot = batch["target"].to(device)

        target_ids = torch.argmax(targets_onehot, dim=1)

        logits = model(images)
        loss = criterion(logits, target_ids)

        total_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == target_ids).sum().item()
        total_samples += images.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc

In [ ]:
train_losses_onehot = []
val_losses_onehot = []
train_accs_onehot = []
val_accs_onehot = []

EPOCHS = 30
patience, patience_counter, best_val_loss, best_model_state = 3, 0, float("inf"), None

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch_onehot(model_onehot, train_loader_onehot, optimizer_onehot, criterion_onehot, device)
    val_loss, val_acc = evaluate_onehot(model_onehot, val_loader_onehot, criterion_onehot, device)
    train_losses_onehot.append(train_loss)
    val_losses_onehot.append(val_loss)
    train_accs_onehot.append(train_acc)  # needed by eval cells
    val_accs_onehot.append(val_acc)      # needed by eval cells
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = {k: v.clone() for k, v in model_onehot.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}.")
            break
if best_model_state: model_onehot.load_state_dict(best_model_state)

Epoch 1/30 | Train Loss: 8.0896 | Train Acc: 0.0002 | Val Loss: 8.0773 | Val Acc: 0.0000
Epoch 2/30 | Train Loss: 8.0835 | Train Acc: 0.0007 | Val Loss: 8.0944 | Val Acc: 0.0000
Epoch 3/30 | Train Loss: 8.0721 | Train Acc: 0.0014 | Val Loss: 8.3597 | Val Acc: 0.0000
Epoch 4/30 | Train Loss: 8.0397 | Train Acc: 0.0014 | Val Loss: 8.1791 | Val Acc: 0.0000
Early stopping at epoch 4.


## 13. Train the TF-IDF Version

In [ ]:
model_tfidf = Axis1Model(
    backbone=SimpleCNNBackbone(feature_dim=FEATURE_DIM),
    head=PredictionHead(input_dim=FEATURE_DIM, output_dim=target_dims["tfidf"])
)

print(model_tfidf)

Axis1Model(
  (backbone): SimpleCNNBackbone(
    (conv): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): ReLU()
      (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (10): ReLU()
      (11): AdaptiveAvgPool2d(output_size=(1, 1))
    )
    (feature_layer): Linear(in_features=256, out_features=256, bias=True)
  )
  (head): PredictionHead(
    (net): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_feat

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

model_tfidf = model_tfidf.to(device)

criterion_tfidf = nn.MSELoss()
optimizer_tfidf = torch.optim.Adam(model_tfidf.parameters(), lr=1e-3)

Using device: cuda


In [ ]:
train_loader_tfidf, val_loader_tfidf, test_loader_tfidf = make_dataloaders("tfidf", batch_size=256)

batch_tfidf = next(iter(train_loader_tfidf))
print("Image batch shape:", batch_tfidf["image"].shape)
print("Target batch shape:", batch_tfidf["target"].shape)
print("First text:", batch_tfidf["text"][0])

Image batch shape: torch.Size([256, 3, 128, 128])
Target batch shape: torch.Size([256, 178])
First text: This picture contains 2 large red dice displaying two and three.


In [ ]:
def train_one_epoch_tfidf(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_samples = 0

    for batch in dataloader:
        images = batch["image"].to(device)
        targets = batch["target"].to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

    avg_loss = total_loss / total_samples
    return avg_loss


@torch.no_grad()
def evaluate_tfidf(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_samples = 0

    for batch in dataloader:
        images = batch["image"].to(device)
        targets = batch["target"].to(device)

        outputs = model(images)
        loss = criterion(outputs, targets)

        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

    avg_loss = total_loss / total_samples
    return avg_loss

In [ ]:
train_losses_tfidf = []
val_losses_tfidf = []

EPOCHS = 30
patience, patience_counter, best_val_loss, best_model_state = 3, 0, float("inf"), None

for epoch in range(EPOCHS):
    train_loss = train_one_epoch_tfidf(model_tfidf, train_loader_tfidf, optimizer_tfidf, criterion_tfidf, device)
    val_loss = evaluate_tfidf(model_tfidf, val_loader_tfidf, criterion_tfidf, device)
    train_losses_tfidf.append(train_loss)
    val_losses_tfidf.append(val_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = {k: v.clone() for k, v in model_tfidf.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}.")
            break
if best_model_state: model_tfidf.load_state_dict(best_model_state)

Epoch 1/30 | Train Loss: 0.0051 | Val Loss: 0.0044
Epoch 2/30 | Train Loss: 0.0044 | Val Loss: 0.0044
Epoch 3/30 | Train Loss: 0.0043 | Val Loss: 0.0043
Epoch 4/30 | Train Loss: 0.0042 | Val Loss: 0.0041
Epoch 5/30 | Train Loss: 0.0042 | Val Loss: 0.0041
Epoch 6/30 | Train Loss: 0.0041 | Val Loss: 0.0041
Epoch 7/30 | Train Loss: 0.0041 | Val Loss: 0.0041
Epoch 8/30 | Train Loss: 0.0041 | Val Loss: 0.0041
Epoch 9/30 | Train Loss: 0.0041 | Val Loss: 0.0040
Epoch 10/30 | Train Loss: 0.0040 | Val Loss: 0.0040
Epoch 11/30 | Train Loss: 0.0040 | Val Loss: 0.0040
Epoch 12/30 | Train Loss: 0.0040 | Val Loss: 0.0039
Epoch 13/30 | Train Loss: 0.0039 | Val Loss: 0.0039
Epoch 14/30 | Train Loss: 0.0039 | Val Loss: 0.0039
Epoch 15/30 | Train Loss: 0.0039 | Val Loss: 0.0038
Epoch 16/30 | Train Loss: 0.0038 | Val Loss: 0.0038
Epoch 17/30 | Train Loss: 0.0038 | Val Loss: 0.0038
Epoch 18/30 | Train Loss: 0.0038 | Val Loss: 0.0038
Epoch 19/30 | Train Loss: 0.0037 | Val Loss: 0.0037
Epoch 20/30 | Train L

## 14. Train the SBERT Version

In [ ]:
model_sbert = Axis1Model(
    backbone=SimpleCNNBackbone(feature_dim=FEATURE_DIM),
    head=PredictionHead(input_dim=FEATURE_DIM, output_dim=target_dims["sbert"])
)

print(model_sbert)

Axis1Model(
  (backbone): SimpleCNNBackbone(
    (conv): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): ReLU()
      (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (10): ReLU()
      (11): AdaptiveAvgPool2d(output_size=(1, 1))
    )
    (feature_layer): Linear(in_features=256, out_features=256, bias=True)
  )
  (head): PredictionHead(
    (net): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_feat

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

model_sbert = model_sbert.to(device)

criterion_sbert = nn.MSELoss()
optimizer_sbert = torch.optim.Adam(model_sbert.parameters(), lr=1e-3)

Using device: cuda


In [ ]:
train_loader_sbert, val_loader_sbert, test_loader_sbert = make_dataloaders("sbert", batch_size=256)

batch_sbert = next(iter(train_loader_sbert))
print("Image batch shape:", batch_sbert["image"].shape)
print("Target batch shape:", batch_sbert["target"].shape)
print("First text:", batch_sbert["text"][0])

Image batch shape: torch.Size([256, 3, 128, 128])
Target batch shape: torch.Size([256, 384])
First text: The image shows a medium yellow die showing three and a medium purple die showing three.


In [ ]:
def train_one_epoch_sbert(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_samples = 0

    for batch in dataloader:
        images = batch["image"].to(device)
        targets = batch["target"].to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

    avg_loss = total_loss / total_samples
    return avg_loss


@torch.no_grad()
def evaluate_sbert(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_samples = 0

    for batch in dataloader:
        images = batch["image"].to(device)
        targets = batch["target"].to(device)

        outputs = model(images)
        loss = criterion(outputs, targets)

        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

    avg_loss = total_loss / total_samples
    return avg_loss

In [ ]:
train_losses_sbert = []
val_losses_sbert = []

EPOCHS = 30
patience, patience_counter, best_val_loss, best_model_state = 3, 0, float("inf"), None

for epoch in range(EPOCHS):
    train_loss = train_one_epoch_sbert(model_sbert, train_loader_sbert, optimizer_sbert, criterion_sbert, device)
    val_loss = evaluate_sbert(model_sbert, val_loader_sbert, criterion_sbert, device)
    train_losses_sbert.append(train_loss)
    val_losses_sbert.append(val_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = {k: v.clone() for k, v in model_sbert.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}.")
            break
if best_model_state: model_sbert.load_state_dict(best_model_state)

Epoch 1/30 | Train Loss: 0.0015 | Val Loss: 0.0007
Epoch 2/30 | Train Loss: 0.0007 | Val Loss: 0.0007
Epoch 3/30 | Train Loss: 0.0007 | Val Loss: 0.0007
Epoch 4/30 | Train Loss: 0.0007 | Val Loss: 0.0007
Epoch 5/30 | Train Loss: 0.0007 | Val Loss: 0.0007
Epoch 6/30 | Train Loss: 0.0007 | Val Loss: 0.0007
Epoch 7/30 | Train Loss: 0.0007 | Val Loss: 0.0006
Epoch 8/30 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 9/30 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 10/30 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 11/30 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 12/30 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 13/30 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 14/30 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 15/30 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 16/30 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 17/30 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 18/30 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 19/30 | Train Loss: 0.0006 | Val Loss: 0.0006
Epoch 20/30 | Train L

## 15. First Axis 1 Results Summary

In [ ]:
axis1_summary = pd.DataFrame([
    {
        "Representation": "One-hot",
        "Train Loss (last)": train_losses_onehot[-1],
        "Val Loss (last)": val_losses_onehot[-1],
        "Train Acc (last)": train_accs_onehot[-1],
        "Val Acc (last)": val_accs_onehot[-1],
        "Notes": "Classification over full caption classes; very hard output space"
    },
    {
        "Representation": "TF-IDF",
        "Train Loss (last)": train_losses_tfidf[-1],
        "Val Loss (last)": val_losses_tfidf[-1],
        "Train Acc (last)": None,
        "Val Acc (last)": None,
        "Notes": "Regression to weighted word vectors"
    },
    {
        "Representation": "SBERT",
        "Train Loss (last)": train_losses_sbert[-1],
        "Val Loss (last)": val_losses_sbert[-1],
        "Train Acc (last)": None,
        "Val Acc (last)": None,
        "Notes": "Regression to dense semantic embeddings"
    }
])

axis1_summary

,Representation,Train Loss (last),Val Loss (last),Train Acc (last),Val Acc (last),Notes
0,One-hot,8.039684,8.179133,0.001429,0.0,Classification over full caption classes; very...
1,TF-IDF,0.003389,0.003358,NaN,NaN,Regression to weighted word vectors
2,SBERT,0.000542,0.000539,NaN,NaN,Regression to dense semantic embeddings


### Initial observations

- The one-hot baseline is trainable but performs very poorly, likely because it treats each full caption as a separate class, creating a very large and sparse output space.
- TF-IDF and SBERT are both more stable than one-hot under the current setup.
- SBERT shows the smoothest loss reduction so far, suggesting that dense semantic targets may be easier for the image model to learn than full-sentence class labels.
- These are still early-stage results; more evaluation metrics are needed before drawing final conclusions.

## 16. Plot Loss Curves

In [ ]:
epochs_onehot = range(1, len(train_losses_onehot) + 1)

plt.figure(figsize=(6, 4))
plt.plot(epochs_onehot, train_losses_onehot, marker='o', label='Train Loss')
plt.plot(epochs_onehot, val_losses_onehot, marker='o', label='Val Loss')
plt.title("One-hot Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.savefig("One-hot Loss Curve.png", dpi=150, bbox_inches="tight")

In [ ]:
epochs_tfidf = range(1, len(train_losses_tfidf) + 1)

plt.figure(figsize=(6, 4))
plt.plot(epochs_tfidf, train_losses_tfidf, marker='o', label='Train Loss')
plt.plot(epochs_tfidf, val_losses_tfidf, marker='o', label='Val Loss')
plt.title("TF-IDF Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.savefig("TF-IDF Loss Curve.png", dpi=150, bbox_inches="tight")

In [ ]:
epochs_sbert = range(1, len(train_losses_sbert) + 1)

plt.figure(figsize=(6, 4))
plt.plot(epochs_sbert, train_losses_sbert, marker='o', label='Train Loss')
plt.plot(epochs_sbert, val_losses_sbert, marker='o', label='Val Loss')
plt.title("SBERT Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.savefig("SBERT Loss Curve.png", dpi=150, bbox_inches="tight")

## 17. Axis 1 Results Table

In [ ]:
axis1_results_table = pd.DataFrame([
    {
        "Representation": "One-hot",
        "Target Dim": target_dims["onehot"],
        "Final Train Loss": train_losses_onehot[-1],
        "Final Val Loss": val_losses_onehot[-1],
        "Final Train Acc": train_accs_onehot[-1],
        "Final Val Acc": val_accs_onehot[-1],
        "Observation": "Trainable, but poor generalisation and very weak classification performance."
    },
    {
        "Representation": "TF-IDF",
        "Target Dim": target_dims["tfidf"],
        "Final Train Loss": train_losses_tfidf[-1],
        "Final Val Loss": val_losses_tfidf[-1],
        "Final Train Acc": None,
        "Final Val Acc": None,
        "Observation": "Stable optimisation; lexical target is easier to learn than full caption classes."
    },
    {
        "Representation": "SBERT",
        "Target Dim": target_dims["sbert"],
        "Final Train Loss": train_losses_sbert[-1],
        "Final Val Loss": val_losses_sbert[-1],
        "Final Train Acc": None,
        "Final Val Acc": None,
        "Observation": "Smoothest and most stable training; semantic target appears easiest to learn."
    }
])

axis1_results_table

,Representation,Target Dim,Final Train Loss,Final Val Loss,Final Train Acc,Final Val Acc,Observation
0,One-hot,3238,8.039684,8.179133,0.001429,0.0,"Trainable, but poor generalisation and very we..."
1,TF-IDF,178,0.003389,0.003358,NaN,NaN,Stable optimisation; lexical target is easier ...
2,SBERT,384,0.000542,0.000539,NaN,NaN,Smoothest and most stable training; semantic t...


## 18. Test Set Evaluation

In [ ]:
@torch.no_grad()
def test_onehot(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in dataloader:
        images = batch["image"].to(device)
        targets_onehot = batch["target"].to(device)

        target_ids = torch.argmax(targets_onehot, dim=1)

        logits = model(images)
        loss = criterion(logits, target_ids)

        total_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == target_ids).sum().item()
        total_samples += images.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc


test_loss_onehot, test_acc_onehot = test_onehot(
    model_onehot, test_loader_onehot, criterion_onehot, device
)

print("One-hot Test Loss:", round(test_loss_onehot, 6))
print("One-hot Test Acc: ", round(test_acc_onehot, 6))

One-hot Test Loss: 8.077539
One-hot Test Acc:  0.0


In [ ]:
@torch.no_grad()
def test_vector_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_cosine = 0.0
    total_samples = 0

    for batch in dataloader:
        images = batch["image"].to(device)
        targets = batch["target"].to(device)

        outputs = model(images)
        loss = criterion(outputs, targets)

        total_loss += loss.item() * images.size(0)

        batch_cosine = F.cosine_similarity(outputs, targets, dim=1).mean().item()
        total_cosine += batch_cosine * images.size(0)

        total_samples += images.size(0)

    avg_loss = total_loss / total_samples
    avg_cosine = total_cosine / total_samples
    return avg_loss, avg_cosine


test_loss_tfidf, test_cosine_tfidf = test_vector_model(
    model_tfidf, test_loader_tfidf, criterion_tfidf, device
)

print("TF-IDF Test Loss:   ", round(test_loss_tfidf, 6))
print("TF-IDF Test Cosine: ", round(test_cosine_tfidf, 6))

TF-IDF Test Loss:    0.003365
TF-IDF Test Cosine:  0.631757


In [ ]:
test_loss_sbert, test_cosine_sbert = test_vector_model(
    model_sbert, test_loader_sbert, criterion_sbert, device
)

print("SBERT Test Loss:   ", round(test_loss_sbert, 6))
print("SBERT Test Cosine: ", round(test_cosine_sbert, 6))

SBERT Test Loss:    0.000539
SBERT Test Cosine:  0.890472


In [ ]:
axis1_test_results = pd.DataFrame([
    {
        "Representation": "One-hot",
        "Test Loss": test_loss_onehot,
        "Test Accuracy": test_acc_onehot,
        "Test Cosine": None,
        "Comment": "Full-caption classification baseline"
    },
    {
        "Representation": "TF-IDF",
        "Test Loss": test_loss_tfidf,
        "Test Accuracy": None,
        "Test Cosine": test_cosine_tfidf,
        "Comment": "Lexical vector target"
    },
    {
        "Representation": "SBERT",
        "Test Loss": test_loss_sbert,
        "Test Accuracy": None,
        "Test Cosine": test_cosine_sbert,
        "Comment": "Dense semantic vector target"
    }
])

for col in ["Test Loss", "Test Accuracy", "Test Cosine"]:
    axis1_test_results[col] = axis1_test_results[col].apply(
        lambda x: round(x, 6) if pd.notnull(x) else x
    )

axis1_test_results

,Representation,Test Loss,Test Accuracy,Test Cosine,Comment
0,One-hot,8.077539,0.0,NaN,Full-caption classification baseline
1,TF-IDF,0.003365,NaN,0.631757,Lexical vector target
2,SBERT,0.000539,NaN,0.890472,Dense semantic vector target


## 19. Final Axis 1 Results Table

In [ ]:
axis1_final_results = pd.DataFrame([
    {
        "Representation": "One-hot",
        "Target Dim": target_dims["onehot"],
        "Final Train Loss": train_losses_onehot[-1],
        "Final Val Loss": val_losses_onehot[-1],
        "Final Train Acc": train_accs_onehot[-1],
        "Final Val Acc": val_accs_onehot[-1],
        "Test Loss": test_loss_onehot,
        "Test Accuracy": test_acc_onehot,
        "Test Cosine": None,
        "Interpretation": "Very weak baseline; treating each full caption as a class makes the output space too sparse and hard to generalise."
    },
    {
        "Representation": "TF-IDF",
        "Target Dim": target_dims["tfidf"],
        "Final Train Loss": train_losses_tfidf[-1],
        "Final Val Loss": val_losses_tfidf[-1],
        "Final Train Acc": None,
        "Final Val Acc": None,
        "Test Loss": test_loss_tfidf,
        "Test Accuracy": None,
        "Test Cosine": test_cosine_tfidf,
        "Interpretation": "More stable than one-hot; lexical target space is easier to learn because related captions share words."
    },
    {
        "Representation": "SBERT",
        "Target Dim": target_dims["sbert"],
        "Final Train Loss": train_losses_sbert[-1],
        "Final Val Loss": val_losses_sbert[-1],
        "Final Train Acc": None,
        "Final Val Acc": None,
        "Test Loss": test_loss_sbert,
        "Test Accuracy": None,
        "Test Cosine": test_cosine_sbert,
        "Interpretation": "Best overall result; dense semantic targets appear easiest for the image model to learn."
    }
])

for col in [
    "Final Train Loss", "Final Val Loss",
    "Final Train Acc", "Final Val Acc",
    "Test Loss", "Test Accuracy", "Test Cosine"
]:
    axis1_final_results[col] = axis1_final_results[col].apply(
        lambda x: round(x, 6) if pd.notnull(x) else x
    )

axis1_final_results

,Representation,Target Dim,Final Train Loss,Final Val Loss,Final Train Acc,Final Val Acc,Test Loss,Test Accuracy,Test Cosine,Interpretation
0,One-hot,3238,8.039684,8.179133,0.001429,0.0,8.077539,0.0,NaN,Very weak baseline; treating each full caption...
1,TF-IDF,178,0.003389,0.003358,NaN,NaN,0.003365,NaN,0.631757,More stable than one-hot; lexical target space...
2,SBERT,384,0.000542,0.000539,NaN,NaN,0.000539,NaN,0.890472,Best overall result; dense semantic targets ap...


### Axis 1 result interpretation

The one-hot baseline performed very poorly. This is expected because it treats each full caption as an independent class, creating a large and sparse output space. As a result, the model struggles to generalise beyond exact caption forms.

TF-IDF performed much more stably than one-hot. This suggests that predicting a lexical vector space is easier than full-caption classification, because captions with similar content share important words and phrases.

SBERT gave the strongest overall results. It achieved the smoothest training dynamics and the highest test cosine similarity, suggesting that dense semantic targets are the most learnable representation under the current setup. This supports the hypothesis that semantically structured embeddings generalise better than sparse sentence-level labels.

# Axis 2 — Multi-Task Image Prediction

Three CNN backbones simultaneously predict all four symbol fields from a single image:
count · face value · colour · size.

**Hypothesis:** Fine-tuned ResNet-18 will outperform a custom CNN because pretrained
spatial features transfer to pip-counting and colour recognition despite the synthetic
domain gap.

**Four parallel prediction heads:**

| Head | Output | Predicts |
|---|---|---|
| `head_count` | `[B, 3]` | number of dice (1/2/3) |
| `head_value` | `[B, 3, 6]` | face value per slot |
| `head_colour` | `[B, 3, 7]` | colour per slot |
| `head_size` | `[B, 3, 3]` | size per slot |

Loss = weighted sum of four CrossEntropy losses with masked slot loss
excluding absent die slots (padded with -1).

**Notebook order:**
1. Imports & Configuration
2. DiceDataset — Four-Task Labels
3. Data Loaders
4. Multi-Task Model Architecture
5. Loss Functions & Prediction Runner
6. Training Loop
7. Run All Experiments
8. Results Table
9. Confusion Matrices
10. Learning Curves
11. slots_to_sentence() — Decode Predictions to Text
12. Evaluate with eval.py

## 1. Imports & Configuration

In [ ]:
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DATASET_ROOT = "dataset"
OUTPUT_DIR   = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")


MAX_DICE           = 3

NUM_DICE_CLASSES   = 3
DICE_CLASS_NAMES   = ["1 die", "2 dice", "3 dice"]

NUM_VALUE_CLASSES  = 6
VALUE_CLASS_NAMES  = ["1 pip", "2 pips", "3 pips", "4 pips", "5 pips", "6 pips"]

VALID_COLOURS      = ["white", "red", "blue", "green", "yellow", "purple", "peach"]
NUM_COLOUR_CLASSES = len(VALID_COLOURS)
COLOUR_CLASS_NAMES = VALID_COLOURS
COLOUR_TO_IDX      = {c: i for i, c in enumerate(VALID_COLOURS)}

VALID_SIZES        = ["small", "medium", "large"]
NUM_SIZE_CLASSES   = len(VALID_SIZES)
SIZE_CLASS_NAMES   = VALID_SIZES
SIZE_TO_IDX        = {s: i for i, s in enumerate(VALID_SIZES)}

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Tasks  : count({NUM_DICE_CLASSES})  value({NUM_VALUE_CLASSES})  "
      f"colour({NUM_COLOUR_CLASSES})  size({NUM_SIZE_CLASSES})")

for split in ["train", "val", "test"]:
    p = Path(DATASET_ROOT) / split / "labels.csv"
    assert p.exists(), f"Missing: {p} — run data generation first."
    df = pd.read_csv(p, nrows=1)
    required = {"image", "sym_num_dice", "sym_values", "sym_colours", "sym_sizes"}
    missing  = required - set(df.columns)
    assert not missing, f"CSV missing columns: {missing}"
print("Dataset schema valid. Ready.")

Device : cuda
PyTorch: 2.10.0+cu128
Tasks  : count(3)  value(6)  colour(7)  size(3)
Dataset schema valid. Ready.


## 2. DiceDataset — Four-Task Labels

Each sample returns four label tensors (all shape `[MAX_DICE]` except count):

| Key | Shape | Padding value | Notes |
|---|---|---|---|
| `sym_num_dice` | scalar | — | 0-indexed (0=1die, 1=2dice, 2=3dice) |
| `sym_values`   | `[3]` | `0` | die face values 1–6, stored as 0-indexed (0–5); pad=`-1` |
| `sym_colours`  | `[3]` | `-1` | colour index 0–6; pad=`-1` |
| `sym_sizes`    | `[3]` | `-1` | size index 0–2; pad=`-1` |

A unified pad of `-1` is used for all per-slot tasks so the masked loss can
apply a single `targets >= 0` mask consistently.


In [ ]:
class DiceDataset(Dataset):
    VALID_SPLITS = ("train", "val", "test")

    def __init__(self, root_dir: str, split: str = "train", transform=None):
        if split not in self.VALID_SPLITS:
            raise ValueError(f"split must be one of {self.VALID_SPLITS}")

        self.image_dir = os.path.join(root_dir, split, "images")
        self.csv_path  = os.path.join(root_dir, split, "labels.csv")
        assert os.path.isdir(self.image_dir), f"Missing image dir: {self.image_dir}"
        assert os.path.isfile(self.csv_path), f"Missing CSV: {self.csv_path}"

        self.samples = []
        with open(self.csv_path, "r", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                self.samples.append(row)
        assert len(self.samples) > 0, "No samples found in CSV."

        self.transform = transform or v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
        ])
        print(f"DiceDataset ({split}): {len(self.samples)} samples loaded.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row      = self.samples[idx]
        img_path = os.path.join(self.image_dir, row["image"])
        image    = Image.open(img_path).convert("RGB")
        image    = self.transform(image)

        num_dice = int(row["sym_num_dice"])

        raw_values = list(map(int, row["sym_values"].split()))
        colours    = row["sym_colours"].split()
        sizes      = row["sym_sizes"].split()

        count_label   = num_dice - 1


        value_labels  = [(v - 1) for v in raw_values]  + [-1] * (MAX_DICE - len(raw_values))
        colour_labels = [COLOUR_TO_IDX[c] for c in colours] + [-1] * (MAX_DICE - len(colours))
        size_labels   = [SIZE_TO_IDX[s]   for s in sizes]   + [-1] * (MAX_DICE - len(sizes))

        label = {
            "sym_num_dice": torch.tensor(count_label,   dtype=torch.long),
            "sym_values":   torch.tensor(value_labels,  dtype=torch.long),
            "sym_colours":  torch.tensor(colour_labels, dtype=torch.long),
            "sym_sizes":    torch.tensor(size_labels,   dtype=torch.long),
            "text":         row.get("text", ""),
        }
        return image, label


def dice_collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    labels = {
        "sym_num_dice": torch.stack([item[1]["sym_num_dice"] for item in batch]),
        "sym_values":   torch.stack([item[1]["sym_values"]   for item in batch]),
        "sym_colours":  torch.stack([item[1]["sym_colours"]  for item in batch]),
        "sym_sizes":    torch.stack([item[1]["sym_sizes"]    for item in batch]),
        "text":         [item[1]["text"]                     for item in batch],
    }
    return images, labels


## 3. Data Loaders

In [ ]:
IMG_SIZE      = 128
BATCH_SIZE    = 256
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = DiceDataset(DATASET_ROOT, "train", train_transform)
val_ds   = DiceDataset(DATASET_ROOT, "val",   eval_transform)
test_ds  = DiceDataset(DATASET_ROOT, "test",  eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=dice_collate_fn, num_workers=N_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=dice_collate_fn, num_workers=N_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=dice_collate_fn, num_workers=N_WORKERS)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


imgs, labels = next(iter(train_loader))
print(f"\nImage batch      : {imgs.shape}")
print(f"sym_num_dice     : {labels['sym_num_dice'].shape}  sample={labels['sym_num_dice'][:4].tolist()}")
print(f"sym_values       : {labels['sym_values'].shape}    sample={labels['sym_values'][0].tolist()}")
print(f"sym_colours      : {labels['sym_colours'].shape}   sample={labels['sym_colours'][0].tolist()}")
print(f"sym_sizes        : {labels['sym_sizes'].shape}     sample={labels['sym_sizes'][0].tolist()}")
print(f"Example text     : {labels['text'][0]}")

counts = Counter(int(row["sym_num_dice"]) for row in train_ds.samples)
print(f"\nTrain class dist : { {DICE_CLASS_NAMES[k-1]: v for k, v in sorted(counts.items())} }")


DiceDataset (train): 4200 samples loaded.
DiceDataset (val): 900 samples loaded.
DiceDataset (test): 900 samples loaded.
Train batches: 17 | Val: 4 | Test: 4

Image batch      : torch.Size([256, 3, 128, 128])
sym_num_dice     : torch.Size([256])  sample=[1, 1, 1, 0]
sym_values       : torch.Size([256, 3])    sample=[1, 1, -1]
sym_colours      : torch.Size([256, 3])   sample=[5, 2, -1]
sym_sizes        : torch.Size([256, 3])     sample=[1, 0, -1]
Example text     : The image shows a medium purple die showing two and a small blue die showing two.

Train class dist : {'1 die': 1416, '2 dice': 1411, '3 dice': 1373}


## 4. Multi-Task Model Architecture

Each backbone feeds a **shared bottleneck**, then splits into **four parallel heads**:

| Head | Output shape | Predicts |
|---|---|---|
| `head_count`  | `[B, 3]` | number of dice (1/2/3) |
| `head_value`  | `[B, 3, 6]` | face value per slot (1–6 pips) |
| `head_colour` | `[B, 3, 7]` | colour per slot (7 options) |
| `head_size`   | `[B, 3, 3]` | size per slot (S/M/L) |

Loss = `λ_count·L_count + λ_value·L_value + λ_colour·L_colour + λ_size·L_size`


In [ ]:
# ── CustomCNN backbone ────────────────────────────────────────────────
class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,   32,  3, padding=1, bias=False), nn.BatchNorm2d(32),  nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32,  64,  3, padding=1, bias=False), nn.BatchNorm2d(64),  nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64,  128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.feature_dim = 256 * 4 * 4

    def forward(self, x):
        return self.features(x).flatten(1)


# ── ResNet-18 backbone ────────────────────────────────────────────────
def build_resnet18(pretrained: bool = True):
    m = models.resnet18(
        weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    )
    feat_dim = m.fc.in_features
    m.fc = nn.Identity()
    return m, feat_dim


# ── EfficientNet-B0 backbone ──────────────────────────────────────────
def build_efficientnet_b0(pretrained: bool = True):
    m = models.efficientnet_b0(
        weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    )
    feat_dim = m.classifier[1].in_features
    m.classifier = nn.Identity()
    return m, feat_dim


# ── Four-task wrapper ─────────────────────────────────────────────────────
class MultiTaskDiceModel(nn.Module):
    """
    Shared backbone → shared bottleneck → four task heads:
      head_count  : [B, NUM_DICE_CLASSES]
      head_value  : [B, MAX_DICE, NUM_VALUE_CLASSES]
      head_colour : [B, MAX_DICE, NUM_COLOUR_CLASSES]
      head_size   : [B, MAX_DICE, NUM_SIZE_CLASSES]
    """
    def __init__(self, backbone: nn.Module, feature_dim: int, dropout: float = 0.4):
        super().__init__()
        self.backbone = backbone
        self.drop     = nn.Dropout(p=dropout)

        self.shared = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
        )

        self.head_count  = nn.Linear(512, NUM_DICE_CLASSES)
        self.head_value  = nn.Linear(512, NUM_VALUE_CLASSES  * MAX_DICE)
        self.head_colour = nn.Linear(512, NUM_COLOUR_CLASSES * MAX_DICE)
        self.head_size   = nn.Linear(512, NUM_SIZE_CLASSES   * MAX_DICE)

    def forward(self, x):
        feat   = self.drop(self.backbone(x))
        shared = self.shared(feat)
        return (
            self.head_count(shared),                                              # [B, 3]
            self.head_value(shared).view(-1, MAX_DICE, NUM_VALUE_CLASSES),        # [B, 3, 6]
            self.head_colour(shared).view(-1, MAX_DICE, NUM_COLOUR_CLASSES),      # [B, 3, 7]
            self.head_size(shared).view(-1, MAX_DICE, NUM_SIZE_CLASSES),          # [B, 3, 3]
        )


def make_model(name: str, pretrained: bool = True) -> MultiTaskDiceModel:
    if name == "CustomCNN":
        bb = CustomCNN(); feat_dim = bb.feature_dim
    elif name == "ResNet-18":
        bb, feat_dim = build_resnet18(pretrained)
    elif name == "EfficientNet-B0":
        bb, feat_dim = build_efficientnet_b0(pretrained)
    else:
        raise ValueError(f"Unknown model: {name}")
    return MultiTaskDiceModel(bb, feat_dim)


# Smoke test
_m = make_model("CustomCNN", pretrained=False)
_x = torch.randn(4, 3, 224, 224)
_lc, _lv, _lcol, _ls = _m(_x)
print(f"count  logits : {_lc.shape}    → expect [4, 3]")
print(f"value  logits : {_lv.shape}  → expect [4, 3, 6]")
print(f"colour logits : {_lcol.shape}  → expect [4, 3, 7]")
print(f"size   logits : {_ls.shape}    → expect [4, 3, 3]")
del _m, _x, _lc, _lv, _lcol, _ls


count  logits : torch.Size([4, 3])    → expect [4, 3]
value  logits : torch.Size([4, 3, 6])  → expect [4, 3, 6]
colour logits : torch.Size([4, 3, 7])  → expect [4, 3, 7]
size   logits : torch.Size([4, 3, 3])    → expect [4, 3, 3]


In [ ]:
LAMBDA_COUNT  = 1.0
LAMBDA_VALUE  = 1.0
LAMBDA_COLOUR = 1.0
LAMBDA_SIZE   = 1.0

_ce = nn.CrossEntropyLoss()


def masked_slot_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    logits  : [B, MAX_DICE, C]
    targets : [B, MAX_DICE]  — -1 = absent slot (excluded from loss)
    """
    mask = targets >= 0
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits.device, requires_grad=True)
    return _ce(logits[mask], targets[mask])


def multitask_loss(lc, lv, lcol, ls, labels):
    """Return (total, l_count, l_value, l_colour, l_size)."""
    l_count  = _ce(lc, labels["sym_num_dice"])
    l_value  = masked_slot_loss(lv,   labels["sym_values"])
    l_colour = masked_slot_loss(lcol, labels["sym_colours"])
    l_size   = masked_slot_loss(ls,   labels["sym_sizes"])
    total    = (LAMBDA_COUNT  * l_count  + LAMBDA_VALUE  * l_value +
                LAMBDA_COLOUR * l_colour + LAMBDA_SIZE   * l_size)
    return total, l_count, l_value, l_colour, l_size


# ── Prediction runner ─────────────────────────────────────────────────────
def get_predictions(model, loader):
    """Return dict of pred/true arrays for all four tasks plus avg loss."""
    model.eval()
    pc, tc, pv, tv, pcol, tcol, ps, ts = [], [], [], [], [], [], [], []
    total_loss, n = 0.0, 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            lbs  = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v)
                    for k, v in labels.items()}
            lc, lv, lcol, ls = model(imgs)
            loss, *_ = multitask_loss(lc, lv, lcol, ls, lbs)
            total_loss += loss.item() * len(imgs); n += len(imgs)

            pc.extend(lc.argmax(1).cpu().numpy())
            tc.extend(lbs["sym_num_dice"].cpu().numpy())
            pv.append(lv.argmax(-1).cpu().numpy())
            tv.append(lbs["sym_values"].cpu().numpy())
            pcol.append(lcol.argmax(-1).cpu().numpy())
            tcol.append(lbs["sym_colours"].cpu().numpy())
            ps.append(ls.argmax(-1).cpu().numpy())
            ts.append(lbs["sym_sizes"].cpu().numpy())

    return {
        "pred_count":  np.array(pc),         "true_count":  np.array(tc),
        "pred_value":  np.vstack(pv),         "true_value":  np.vstack(tv),
        "pred_colour": np.vstack(pcol),        "true_colour": np.vstack(tcol),
        "pred_size":   np.vstack(ps),          "true_size":   np.vstack(ts),
        "avg_loss":    total_loss / n,
    }


def masked_metrics(true_arr, pred_arr, n_classes):
    """Flatten [N, MAX_DICE], drop pad slots (-1), return metric dict."""
    ft = true_arr.flatten(); fp = pred_arr.flatten()
    mask = ft >= 0; ft, fp = ft[mask], fp[mask]
    labs = list(range(n_classes))
    return {
        "accuracy":  round(accuracy_score(ft, fp), 4),
        "f1_macro":  round(f1_score(ft, fp, average="macro", zero_division=0, labels=labs), 4),
        "precision": round(precision_score(ft, fp, average="macro", zero_division=0, labels=labs), 4),
        "recall":    round(recall_score(ft, fp, average="macro", zero_division=0, labels=labs), 4),
        "cm":        confusion_matrix(ft, fp, labels=labs),
        "flat_true": ft, "flat_pred": fp,
    }


def compute_all_metrics(preds):
    return {
        "count":  {
            "accuracy":  round(accuracy_score(preds["true_count"], preds["pred_count"]), 4),
            "f1_macro":  round(f1_score(preds["true_count"], preds["pred_count"],
                               average="macro", zero_division=0), 4),
            "precision": round(precision_score(preds["true_count"], preds["pred_count"],
                               average="macro", zero_division=0), 4),
            "recall":    round(recall_score(preds["true_count"], preds["pred_count"],
                               average="macro", zero_division=0), 4),
            "cm":        confusion_matrix(preds["true_count"], preds["pred_count"], labels=[0,1,2]),
        },
        "value":  masked_metrics(preds["true_value"],  preds["pred_value"],  NUM_VALUE_CLASSES),
        "colour": masked_metrics(preds["true_colour"], preds["pred_colour"], NUM_COLOUR_CLASSES),
        "size":   masked_metrics(preds["true_size"],   preds["pred_size"],   NUM_SIZE_CLASSES),
    }


## 6. Training Loop

In [ ]:
def train_model(model, name, epochs=30, lr=1e-3, weight_decay=1e-4, patience=5):
    model     = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc     = 0.0
    best_val_loss    = float("inf")
    best_state       = None
    patience_counter = 0
    epoch_log        = []
    t0               = time.time()

    print(f"\n{'='*68}")
    print(f"  Training : {name}  |  Epochs: {epochs}  |  LR: {lr}  |  Device: {DEVICE}")
    print(f"{'='*68}")
    print(f"{'Epoch':>6}  {'TrainAcc':>9}  {'ValAcc(cnt)':>11}  {'ValLoss':>9}")

    for ep in range(1, epochs + 1):
        model.train()
        correct, n = 0, 0
        for imgs, labels in train_loader:
            imgs = imgs.to(DEVICE)
            lbs  = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v)
                    for k, v in labels.items()}
            optimizer.zero_grad()
            lc, lv, lcol, ls = model(imgs)
            loss, *_ = multitask_loss(lc, lv, lcol, ls, lbs)
            loss.backward()
            optimizer.step()
            correct += (lc.argmax(1) == lbs["sym_num_dice"]).sum().item()
            n       += len(imgs)
        train_acc = correct / n

        val_preds = get_predictions(model, val_loader)
        val_acc   = accuracy_score(val_preds["true_count"], val_preds["pred_count"])
        val_loss  = val_preds["avg_loss"]
        scheduler.step()

        epoch_log.append({"epoch": ep, "train_acc": round(train_acc, 4),
                          "val_acc": round(val_acc, 4), "val_loss": round(val_loss, 4)})


        print(f"{ep:>6}  {train_acc:>9.4f}  {val_acc:>11.4f}  {val_loss:>9.4f}")

        # Early stopping on val loss, save best on val acc
        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n  Early stopping at epoch {ep} — val loss stalled for {patience} epochs.")
                break

        if val_acc >= best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    test_preds   = get_predictions(model, test_loader)
    val_preds2   = get_predictions(model, val_loader)
    elapsed      = time.time() - t0
    test_metrics = compute_all_metrics(test_preds)
    val_metrics  = compute_all_metrics(val_preds2)

    print(f"\n  Best val acc (count) : {best_val_acc:.4f}   |   Time: {elapsed:.1f}s")
    for task, names in [("count", DICE_CLASS_NAMES), ("value", VALUE_CLASS_NAMES),
                         ("colour", COLOUR_CLASS_NAMES), ("size", SIZE_CLASS_NAMES)]:
        tm = test_metrics[task]
        print(f"  Test [{task:6s}]  Acc: {tm['accuracy']:.4f}   F1: {tm['f1_macro']:.4f}")

    print("\n--- Count (test) ---")
    print(classification_report(test_preds["true_count"], test_preds["pred_count"],
                                target_names=DICE_CLASS_NAMES, digits=3))

    for task, class_names, n_cls, tk, pk in [
        ("Value",  VALUE_CLASS_NAMES,  NUM_VALUE_CLASSES,  "true_value",  "pred_value"),
        ("Colour", COLOUR_CLASS_NAMES, NUM_COLOUR_CLASSES, "true_colour", "pred_colour"),
        ("Size",   SIZE_CLASS_NAMES,   NUM_SIZE_CLASSES,   "true_size",   "pred_size"),
    ]:
        ft = test_preds[tk].flatten(); fp = test_preds[pk].flatten()
        mask = ft >= 0
        print(f"\n--- {task} (test, valid slots only) ---")
        print(classification_report(ft[mask], fp[mask],
                                    target_names=class_names,
                                    labels=list(range(n_cls)), digits=3))

    return {
        "log":          epoch_log,
        "val_metrics":  val_metrics,
        "test_metrics": test_metrics,
        "test_preds":   test_preds,
        "best_val_acc": best_val_acc,
        "train_time_s": elapsed,
    }


## 7. Run All Experiments

In [ ]:
torch.manual_seed(SEED)

experiments = [
    ("CustomCNN",       make_model("CustomCNN",       pretrained=False), 30, 1e-3, 1e-4),
    ("ResNet-18",       make_model("ResNet-18",        pretrained=True),  30, 5e-4, 1e-4),
    ("EfficientNet-B0", make_model("EfficientNet-B0",  pretrained=True),  30, 5e-4, 1e-4),
]

all_results = {}
for name, model, epochs, lr, wd in experiments:
    torch.manual_seed(SEED)
    result = train_model(model, name, epochs=epochs, lr=lr, weight_decay=wd)
    all_results[name] = result

    log_path = OUTPUT_DIR / f"multitask_log_{name.replace(' ','_').replace('-','')}.csv"
    pd.DataFrame(result["log"]).to_csv(log_path, index=False)
    print(f"  Epoch log → {log_path}\n")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 169MB/s]



  Training : CustomCNN  |  Epochs: 30  |  LR: 0.001  |  Device: cuda
 Epoch   TrainAcc  ValAcc(cnt)    ValLoss
     1     0.6979       0.3267     8.2694
     2     0.9143       0.3311     6.5961
     3     0.9698       0.4844     5.8392
     4     0.9883       0.9844     4.8479
     5     0.9907       0.9978     4.6992
     6     0.9890       0.7644     5.1643
     7     0.9902       0.9989     4.5374
     8     0.9931       0.9744     4.6059
     9     0.9900       0.8633     4.7321
    10     0.9945       0.9989     4.3503
    11     0.9926       0.9133     5.0460
    12     0.9940       0.9967     4.2123
    13     0.9900       1.0000     4.0472
    14     0.9919       1.0000     4.1171
    15     0.9936       1.0000     4.1270
    16     0.9929       0.9989     3.8620
    17     0.9924       1.0000     3.8422
    18     0.9931       1.0000     3.8509
    19     0.9943       1.0000     3.7920
    20     0.9943       1.0000     3.7999
    21     0.9945       1.0000     3.7802
    22

## 8. Results Table — All Four Tasks

In [ ]:
rows = []
for name, result in all_results.items():
    tm = result["test_metrics"]
    rows.append({
        "Model":        name,
        "Pretrained":   "No" if name == "CustomCNN" else "Yes (ImageNet)",
        "Count Acc":    tm["count"]["accuracy"],   "Count F1":  tm["count"]["f1_macro"],
        "Value Acc":    tm["value"]["accuracy"],   "Value F1":  tm["value"]["f1_macro"],
        "Colour Acc":   tm["colour"]["accuracy"],  "Colour F1": tm["colour"]["f1_macro"],
        "Size Acc":     tm["size"]["accuracy"],    "Size F1":   tm["size"]["f1_macro"],
        "Train Time(s)": round(result["train_time_s"], 1),
    })

df = pd.DataFrame(rows)
print("=" * 100)
print("  MULTI-TASK RESULTS — TEST SET (Count · Value · Colour · Size)")
print("=" * 100)
print(df.to_string(index=False))

df.to_csv(OUTPUT_DIR / "multitask_results.csv", index=False)
print(f"\nSaved → {OUTPUT_DIR / 'multitask_results.csv'}")


  MULTI-TASK RESULTS — TEST SET (Count · Value · Colour · Size)
          Model     Pretrained  Count Acc  Count F1  Value Acc  Value F1  Colour Acc  Colour F1  Size Acc  Size F1  Train Time(s)
      CustomCNN             No        1.0       1.0     0.4809    0.4583      0.3394     0.3112    0.6029   0.5887          611.9
      ResNet-18 Yes (ImageNet)        1.0       1.0     0.6123    0.6101      0.5824     0.5817    0.6750   0.6722          417.4
EfficientNet-B0 Yes (ImageNet)        1.0       1.0     0.5979    0.5972      0.5929     0.5926    0.6728   0.6718          506.3

Saved → outputs/multitask_results.csv


In [ ]:
BG      = "#0f172a"
TEXT    = "#e2e8f0"
MUTED   = "#64748b"
PALETTE = {
    "CustomCNN":       "#3b82f6",
    "ResNet-18":       "#22c55e",
    "EfficientNet-B0": "#f59e0b",
}

TASKS = [
    ("count",  "Count (1/2/3)",     DICE_CLASS_NAMES,   None,             None),
    ("value",  "Face Value (1–6)",  VALUE_CLASS_NAMES,  "true_value",     "pred_value"),
    ("colour", "Colour (7 classes)",COLOUR_CLASS_NAMES, "true_colour",    "pred_colour"),
    ("size",   "Size (S/M/L)",      SIZE_CLASS_NAMES,   "true_size",      "pred_size"),
]

n_models, n_tasks = len(all_results), len(TASKS)
fig, axes = plt.subplots(n_models, n_tasks,
                         figsize=(5.5 * n_tasks, 4.5 * n_models),
                         facecolor=BG)
fig.suptitle("Multi-Task Confusion Matrices — Test Set  (Count · Value · Colour · Size)",
             color="white", fontsize=13, fontweight="bold", y=1.01)

for row_i, (model_name, result) in enumerate(all_results.items()):
    c     = PALETTE[model_name]
    preds = result["test_preds"]

    for col_j, (task_key, title, class_names, tk, pk) in enumerate(TASKS):
        ax = axes[row_i][col_j]

        if task_key == "count":
            y_true, y_pred = preds["true_count"], preds["pred_count"]
            labs = [0, 1, 2]
        else:
            ft = preds[tk].flatten(); fp = preds[pk].flatten()
            mask = ft >= 0; y_true, y_pred = ft[mask], fp[mask]
            labs = list(range(len(class_names)))

        cm_raw  = confusion_matrix(y_true, y_pred, labels=labs)
        cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True).clip(min=1)
        tm      = result["test_metrics"][task_key]

        sns.heatmap(
            cm_norm, ax=ax,
            cmap=sns.light_palette(c, as_cmap=True),
            annot=cm_raw, fmt="d",
            linewidths=0.4, linecolor="#334155",
            xticklabels=class_names, yticklabels=class_names,
            cbar=False,
            annot_kws={"size": 8, "weight": "bold", "color": "white"},
        )
        ax.set_facecolor(BG)
        ax.set_title(
            f"{model_name} | {title}\nAcc {tm['accuracy']:.3f}   F1 {tm['f1_macro']:.3f}",
            color=c, fontsize=9, fontweight="bold", pad=7
        )
        ax.set_xlabel("Predicted", color=MUTED, fontsize=7)
        ax.set_ylabel("True",      color=MUTED, fontsize=7)
        ax.tick_params(colors=MUTED, labelsize=6)
        for sp in ax.spines.values(): sp.set_edgecolor("#334155")

plt.tight_layout()
cm_path = OUTPUT_DIR / "multitask_confusion_matrices.png"
fig.savefig(cm_path, dpi=130, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved → {cm_path}")


Saved → outputs/multitask_confusion_matrices.png


## 10. Learning Curves & Comparative Bar Charts

In [ ]:
fig = plt.figure(figsize=(19, 12), facecolor=BG)
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.48, wspace=0.33,
                         left=0.05, right=0.97, top=0.88, bottom=0.09)

def style_ax(ax):
    ax.set_facecolor(BG)
    for sp in ax.spines.values(): sp.set_edgecolor("#334155")
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.yaxis.grid(True, color="#1e293b", linestyle="--", lw=0.6)
    ax.set_axisbelow(True)
    ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)

# Row 0: per-model learning curves (count-task val accuracy)
for col, (name, result) in enumerate(all_results.items()):
    ax  = fig.add_subplot(gs[0, col]); style_ax(ax)
    log = result["log"]
    ep  = [r["epoch"] for r in log]
    c   = PALETTE[name]
    ax.plot(ep, [r["train_acc"] for r in log], color=c, lw=2,             label="Train acc")
    ax.plot(ep, [r["val_acc"]   for r in log], color=c, lw=2, ls="--", alpha=0.7, label="Val acc")
    ax.axhline(result["test_metrics"]["count"]["accuracy"],
               color="#ef4444", lw=1.2, ls=":", label=f"Test {result['test_metrics']['count']['accuracy']:.3f}")
    ax.set_ylim(0.2, 1.10)
    ax.set_xlabel("Epoch", fontsize=9); ax.set_ylabel("Count Accuracy", fontsize=9)
    ax.set_title(name, color=c, fontsize=11, fontweight="bold", pad=7)
    ax.legend(fontsize=7, facecolor="#1e293b", edgecolor="#334155", labelcolor="#cbd5e1")

# Row 0 col 3: F1 heatmap across models × tasks
ax_h = fig.add_subplot(gs[0, 3]); style_ax(ax_h)
task_keys   = ["count", "value", "colour", "size"]
task_labels = ["Count", "Value", "Colour", "Size"]
model_names = list(all_results.keys())
f1_matrix   = np.array([[all_results[m]["test_metrics"][t]["f1_macro"]
                          for t in task_keys] for m in model_names])
sns.heatmap(f1_matrix, ax=ax_h,
            cmap="YlGn", vmin=0, vmax=1,
            annot=True, fmt=".3f", annot_kws={"size": 9, "weight": "bold"},
            xticklabels=task_labels, yticklabels=model_names,
            linewidths=0.5, linecolor="#334155", cbar=False)
ax_h.set_facecolor(BG)
ax_h.set_title("F1 (macro) — All Tasks", color=TEXT, fontsize=10, fontweight="bold", pad=7)
ax_h.tick_params(colors=MUTED, labelsize=8)
for sp in ax_h.spines.values(): sp.set_edgecolor("#334155")

# Row 1: grouped bar chart per task (Accuracy)
for col, (task_key, task_label) in enumerate(zip(task_keys, task_labels)):
    ax = fig.add_subplot(gs[1, col]); style_ax(ax)
    model_list = list(all_results.keys())
    x = np.arange(len(model_list)); w = 0.35
    accs = [all_results[m]["test_metrics"][task_key]["accuracy"] for m in model_list]
    f1s  = [all_results[m]["test_metrics"][task_key]["f1_macro"]  for m in model_list]
    bars_acc = ax.bar(x - w/2, accs, w, label="Accuracy", zorder=3,
                      color=[PALETTE[m] for m in model_list], alpha=0.9)
    bars_f1  = ax.bar(x + w/2, f1s,  w, label="F1 macro", zorder=3,
                      color=[PALETTE[m] for m in model_list], alpha=0.5)
    for bar, v in list(zip(bars_acc, accs)) + list(zip(bars_f1, f1s)):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.008, f"{v:.2f}",
                ha="center", va="bottom", color="white", fontsize=6.5, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace("EfficientNet-", "Eff-") for m in model_list],
                       fontsize=7, color=MUTED)
    ax.set_ylim(0, 1.22); ax.set_ylabel("Score", fontsize=8)
    ax.set_title(f"{task_label} Task — Test Set", color=TEXT, fontsize=10, fontweight="bold", pad=7)
    if col == 0:
        ax.legend(fontsize=7, facecolor="#1e293b", edgecolor="#334155", labelcolor="#cbd5e1")

fig.suptitle("Multi-Task Dice — Count · Face Value · Colour · Size  |  CustomCNN vs ResNet-18 vs EfficientNet-B0",
             color="white", fontsize=12, fontweight="bold", y=0.96)
curves_path = OUTPUT_DIR / "multitask_learning_curves.png"
fig.savefig(curves_path, dpi=130, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved → {curves_path}")


Saved → outputs/multitask_learning_curves.png


## 11. slots_to_sentence() — Decode Predictions to Text

Converts the multi-task model's predicted slot indices back into a natural language
sentence using the same vocabulary and template logic as the dataset generator.

This is the bridge between the multi-task classifier and the shared `eval.py`
evaluation framework.

In [ ]:
_NUMBER_WORDS = {1:"one", 2:"two", 3:"three", 4:"four", 5:"five", 6:"six"}
_COUNT_WORDS  = {1:"one die", 2:"two dice", 3:"three dice"}
_IDX_TO_COLOUR = VALID_COLOURS          # list defined in Axis 2 imports
_IDX_TO_SIZE   = VALID_SIZES            # list defined in Axis 2 imports
_IDX_TO_VALUE  = list(range(1, 7))      # 0-indexed → 1-6 face value


def slots_to_sentence(
    num_dice:  int,
    values:    list,   # list of ints, 1-indexed face values, length = num_dice
    colours:   list,   # list of colour strings, length = num_dice
    sizes:     list,   # list of size strings,   length = num_dice
    template:  int = None,  # 1-4, or None for random
) -> str:
    """
    Reconstruct a natural language sentence from predicted symbol slots.

    Mirrors describe_symbol() from the dataset generator so that decoded
    sentences are in-distribution with the training data.

    Args:
        num_dice:  number of dice in scene (1-3)
        values:    face values for each die (1-6)
        colours:   colour label for each die
        sizes:     size label for each die
        template:  which template variant to use (1-4), or None for random

    Returns:
        A single sentence string.

    Example:
        slots_to_sentence(2, [3, 5], ["blue", "red"], ["large", "small"])
        -> "There are two dice showing three and five; a large blue and a small red."
    """
    if not (1 <= num_dice <= 3):
        raise ValueError(f"num_dice must be 1-3, got {num_dice}")
    if len(values) != num_dice or len(colours) != num_dice or len(sizes) != num_dice:
        raise ValueError("values, colours, sizes must all have length == num_dice")

    def _val(v):
        return _NUMBER_WORDS.get(v, str(v))

    def _join(items):
        if len(items) == 1: return items[0]
        if len(items) == 2: return items[0] + " and " + items[1]
        return ", ".join(items[:-1]) + " and " + items[-1]

    t = template if template is not None else random.randint(1, 4)

    all_same_colour = len(set(colours)) == 1
    all_same_size   = len(set(sizes)) == 1

    if num_dice == 1:
        s, c, v = sizes[0], colours[0], _val(values[0])
        opts = [
            f"There is one {s} {c} die showing {v}.",
            f"The image shows one {s} {c} die with value {v}.",
            f"This picture contains a {s} {c} die displaying {v}.",
            f"A {s} {c} die is shown with a value of {v}.",
        ]
        return opts[t - 1]

    value_words = [_val(v) for v in values]

    if all_same_colour and all_same_size:
        s, c = sizes[0], colours[0]
        count_w = _COUNT_WORDS[num_dice]
        val_str = _join(value_words)
        opts = [
            f"There are {count_w} {s} {c} dice showing {val_str}.",
            f"The image shows {count_w} {s} {c} dice with values {val_str}.",
            f"This picture contains {count_w} {s} {c} dice displaying {val_str}.",
            f"There are {count_w} {s} {c} dice; they show {val_str}.",
        ]
        return opts[t - 1]

    # Mixed: describe each die individually
    phrases = [f"a {sizes[i]} {colours[i]} die showing {_val(values[i])}"
               for i in range(num_dice)]
    dice_str = _join(phrases)
    opts = [
        f"There are {_COUNT_WORDS[num_dice]}: {dice_str}.",
        f"The image shows {dice_str}.",
        f"This picture contains {dice_str}.",
        f"The dice are: {dice_str}.",
    ]
    return opts[t - 1]


def decode_multitask_batch(
    pred_count:  "np.ndarray",   # [B]   0-indexed count class
    pred_value:  "np.ndarray",   # [B, 3] 0-indexed value class (-1 = pad)
    pred_colour: "np.ndarray",   # [B, 3] 0-indexed colour class (-1 = pad)
    pred_size:   "np.ndarray",   # [B, 3] 0-indexed size class (-1 = pad)
    template:    int = None,
) -> list:
    """
    Convert a batch of multi-task predictions to a list of sentence strings.
    Pads beyond num_dice are ignored using the pred_count as the authoritative
    die count.

    Returns:
        List of sentence strings, one per sample.
    """
    sentences = []
    for b in range(len(pred_count)):
        n = int(pred_count[b]) + 1          # 0-indexed → 1/2/3
        n = max(1, min(3, n))               # clamp to valid range

        vals    = [_IDX_TO_VALUE[int(pred_value[b, i])]
                   for i in range(n)]
        colours = [_IDX_TO_COLOUR[int(pred_colour[b, i])]
                   for i in range(n)]
        sizes   = [_IDX_TO_SIZE[int(pred_size[b, i])]
                   for i in range(n)]

        sentences.append(slots_to_sentence(n, vals, colours, sizes, template))
    return sentences


# ── Sanity check ─────────────────────────────────────────────────────────────
print("slots_to_sentence() examples:")
print(slots_to_sentence(1, [3], ["blue"], ["large"]))
print(slots_to_sentence(2, [3, 5], ["blue", "red"], ["large", "small"]))
print(slots_to_sentence(3, [1, 4, 6], ["white", "green", "purple"],
                        ["small", "medium", "large"]))
print()
print("All four templates for a single blue die showing two:")
for t in [1, 2, 3, 4]:
    print(f"  [{t}] {slots_to_sentence(1, [2], ['blue'], ['medium'], template=t)}")

slots_to_sentence() examples:
There is one large blue die showing three.
There are two dice: a large blue die showing three and a small red die showing five.
This picture contains a small white die showing one, a medium green die showing four and a large purple die showing six.

All four templates for a single blue die showing two:
  [1] There is one medium blue die showing two.
  [2] The image shows one medium blue die with value two.
  [3] This picture contains a medium blue die displaying two.
  [4] A medium blue die is shown with a value of two.


## 12. Evaluate with eval.py — Unified Metrics

Runs the full Axis 2 test set through `slots_to_sentence()` then passes decoded
sentences to `evaluate_batch()` from the shared `eval.py` framework.

This produces four metrics consistent with Axis 1 and the LLM comparison:
- **count_correct** — did the model get the number of dice right?
- **colours_correct** — did it get all colours right?
- **values_correct** — did it get all face values right?
- **cosine_sim** — SBERT cosine similarity between predicted and gold sentence
- **bleu** — sentence-level BLEU score

> **Prerequisite:** `eval.py` must be in the same directory as this notebook.
> Run `pip install sentence-transformers nltk` if not already installed.

In [ ]:
def axis2_eval_with_evalpy(model_name: str, result: dict) -> dict:
    """
    Decode all test set predictions to sentences and evaluate with eval.py.

    Args:
        model_name: string label for printing
        result:     result dict from train_model(), containing test_preds

    Returns:
        Dict of averaged metrics from evaluate_batch()
    """
    preds = result["test_preds"]

    # Decode predictions → sentence strings
    decoded = decode_multitask_batch(
        pred_count  = preds["pred_count"],
        pred_value  = preds["pred_value"],
        pred_colour = preds["pred_colour"],
        pred_size   = preds["pred_size"],
        template    = None,   # random template per sample
    )

    # Ground truth sentences — read from test CSV
    test_gold = pd.read_csv(Path(DATASET_ROOT) / "test" / "labels.csv")["text"].tolist()

    assert len(decoded) == len(test_gold), (
        f"Prediction count {len(decoded)} != gold count {len(test_gold)}"
    )

    metrics = evaluate_batch(decoded, test_gold)
    return metrics


print("Running eval.py evaluation on all Axis 2 models...")
print()

axis2_evalpy_results = {}
for name, result in all_results.items():
    metrics = axis2_eval_with_evalpy(name, result)
    axis2_evalpy_results[name] = metrics
    print(f"  {name}")
    for k, v in metrics.items():
        print(f"    {k:<20}: {v:.4f}")
    print()

# ── Summary table ──────────────────────────────────────────────────────────
evalpy_rows = []
for name, m in axis2_evalpy_results.items():
    evalpy_rows.append({
        "Model":           name,
        "Count correct":   round(m["count_correct"],   4),
        "Colours correct": round(m["colours_correct"], 4),
        "Values correct":  round(m["values_correct"],  4),
        "Fully correct":   round(m["fully_correct"],   4),
        "Cosine sim":      round(m["cosine_sim"],       4),
        "BLEU":            round(m["bleu"],             4),
    })

evalpy_df = pd.DataFrame(evalpy_rows)
print("=" * 80)
print("  AXIS 2 — eval.py UNIFIED METRICS — TEST SET")
print("=" * 80)
print(evalpy_df.to_string(index=False))

evalpy_df.to_csv(OUTPUT_DIR / "axis2_evalpy_results.csv", index=False)
print(f"\nSaved → {OUTPUT_DIR / 'axis2_evalpy_results.csv'}")

Running eval.py evaluation on all Axis 2 models...

  CustomCNN
    count_correct       : 0.4989
    colours_correct     : 0.2622
    values_correct      : 0.2578
    sizes_correct       : 0.4611
    fully_correct       : 0.1044
    cosine_sim          : 0.8010
    bleu                : 0.2420

  ResNet-18
    count_correct       : 0.6233
    colours_correct     : 0.5911
    values_correct      : 0.3600
    sizes_correct       : 0.5700
    fully_correct       : 0.2200
    cosine_sim          : 0.8725
    bleu                : 0.3342

  EfficientNet-B0
    count_correct       : 0.6800
    colours_correct     : 0.7033
    values_correct      : 0.4367
    sizes_correct       : 0.6378
    fully_correct       : 0.2856
    cosine_sim          : 0.8840
    bleu                : 0.4037

  AXIS 2 — eval.py UNIFIED METRICS — TEST SET
          Model  Count correct  Colours correct  Values correct  Fully correct  Cosine sim   BLEU
      CustomCNN         0.4989           0.2622          0.2578   

## 13. Decoded Sentence Examples

Qualitative inspection: what does each model actually predict for a sample of test images?

In [ ]:
# Print 10 decoded predictions vs ground truth for best model
best_model_name = max(axis2_evalpy_results,
                      key=lambda n: axis2_evalpy_results[n]["fully_correct"])

print(f"Decoded examples — best model: {best_model_name}")
print("=" * 80)

best_preds   = all_results[best_model_name]["test_preds"]
test_gold_df = pd.read_csv(Path(DATASET_ROOT) / "test" / "labels.csv")
gold_sentences = test_gold_df["text"].tolist()

decoded_best = decode_multitask_batch(
    pred_count  = best_preds["pred_count"],
    pred_value  = best_preds["pred_value"],
    pred_colour = best_preds["pred_colour"],
    pred_size   = best_preds["pred_size"],
    template    = 1,
)

for i in range(10):
    print(f"[{i:02d}] GT  : {gold_sentences[i]}")
    print(f"      Pred: {decoded_best[i]}")
    print()

Decoded examples — best model: EfficientNet-B0
[00] GT  : There is one medium yellow die showing six.
      Pred: There is one small yellow die showing six.

[01] GT  : This picture contains a large yellow die displaying three.
      Pred: There is one large yellow die showing three.

[02] GT  : This picture contains a medium purple die displaying four.
      Pred: There is one medium purple die showing four.

[03] GT  : A large red die is shown with a value of six.
      Pred: There is one large red die showing six.

[04] GT  : This picture contains a large green die displaying six.
      Pred: There is one large green die showing six.

[05] GT  : A medium purple die is shown with a value of five.
      Pred: There is one medium purple die showing five.

[06] GT  : The scene contains a small yellow die showing five and a large blue die showing five.
      Pred: There are two dice: a medium blue die showing five and a small yellow die showing five.

[07] GT  : There are a medium blue d

## 14. Save All Axis Results for Error Analysis

In [ ]:
# Save Axis 1 final results
axis1_final_results.to_csv(OUTPUT_DIR / "axis1_results.csv", index=False)
print(f"Axis 1 results saved → {OUTPUT_DIR / 'axis1_results.csv'}")

# axis2 multitask results already saved in cell 14 (multitask_results.csv)
# axis2 evalpy results already saved above (axis2_evalpy_results.csv)

# Save decoded test sentences for all Axis 2 models
for name, result in all_results.items():
    preds = result["test_preds"]
    decoded = decode_multitask_batch(
        pred_count  = preds["pred_count"],
        pred_value  = preds["pred_value"],
        pred_colour = preds["pred_colour"],
        pred_size   = preds["pred_size"],
        template    = 1,
    )
    out_df = pd.DataFrame({
        "gold":    pd.read_csv(Path(DATASET_ROOT) / "test" / "labels.csv")["text"].tolist(),
        "decoded": decoded,
    })
    safe_name = name.replace(" ", "_").replace("-", "")
    out_df.to_csv(OUTPUT_DIR / f"axis2_decoded_{safe_name}.csv", index=False)
    print(f"Decoded sentences saved → axis2_decoded_{safe_name}.csv")

print("\nAll outputs saved.")

Axis 1 results saved → outputs/axis1_results.csv
Decoded sentences saved → axis2_decoded_CustomCNN.csv
Decoded sentences saved → axis2_decoded_ResNet18.csv
Decoded sentences saved → axis2_decoded_EfficientNetB0.csv

All outputs saved.


In [ ]:
def _decode_slots(pred_count_idx, pred_values, pred_colours, pred_sizes):
    """Convert predicted slot arrays (0-indexed numpy) to readable Python lists."""
    n       = pred_count_idx + 1                           # 0-indexed → 1,2,3 dice
    vals    = [_IDX_TO_VALUE[v]  for v in pred_values[:n]]
    colours = [_IDX_TO_COLOUR[c] for c in pred_colours[:n]]
    sizes   = [_IDX_TO_SIZE[s]   for s in pred_sizes[:n]]
    return n, vals, colours, sizes


def _decode_gt_slots(true_count_idx, true_values, true_colours, true_sizes):
    """Same for ground-truth (sym_values are stored 0-indexed → add 1 for face value)."""
    n       = true_count_idx + 1
    vals    = [v + 1             for v in true_values[:n]]
    colours = [_IDX_TO_COLOUR[c] for c in true_colours[:n]]
    sizes   = [_IDX_TO_SIZE[s]   for s in true_sizes[:n]]
    return n, vals, colours, sizes

In [ ]:
FAILURE_ORDER = [
    "Wrong count",
    "Wrong value only",
    "Wrong colour only",
    "Wrong size only",
    "Wrong value + colour",
    "Wrong value + size",
    "Wrong colour + size",
    "Multiple attributes wrong",
]


def categorise_failures(preds):
    """
    Categorise every incorrect test prediction into a failure mode.

    Priority: if count is wrong that dominates; otherwise we look at which
    per-slot attributes (value / colour / size) differ from ground truth.

    Returns a list of dicts — one per *incorrect* sample:
        idx, failure_mode, gt_count, pred_count, gt_text, pred_text, image_path
    """
    failures  = []
    test_rows = test_ds.samples          # raw CSV rows — keeps image filenames

    for i in range(len(preds["true_count"])):
        tc   = int(preds["true_count"][i])
        pc   = int(preds["pred_count"][i])
        tv   = preds["true_value"][i]    # shape [MAX_DICE]
        pv   = preds["pred_value"][i]
        tcol = preds["true_colour"][i]
        pcol = preds["pred_colour"][i]
        ts   = preds["true_size"][i]
        ps   = preds["pred_size"][i]

        gt_n,   gt_vals,   gt_cols,   gt_szs  = _decode_gt_slots(tc, tv,  tcol, ts)
        pred_n, pred_vals, pred_cols, pred_szs = _decode_slots(   pc, pv,  pcol, ps)

        gt_sent   = slots_to_sentence(gt_n,   gt_vals,   gt_cols,   gt_szs,   template=1)
        pred_sent = slots_to_sentence(pred_n, pred_vals, pred_cols, pred_szs, template=1)

        if gt_sent == pred_sent:
            continue                          # fully correct — skip

        # Assign failure mode
        if tc != pc:
            mode = "Wrong count"
        else:
            n          = gt_n
            val_wrong  = any(int(tv[j]) != int(pv[j])   for j in range(n))
            col_wrong  = any(int(tcol[j]) != int(pcol[j]) for j in range(n))
            size_wrong = any(int(ts[j]) != int(ps[j])   for j in range(n))

            wrong = (val_wrong, col_wrong, size_wrong)
            mode  = {
                (True,  True,  True ): "Multiple attributes wrong",
                (True,  True,  False): "Wrong value + colour",
                (True,  False, True ): "Wrong value + size",
                (False, True,  True ): "Wrong colour + size",
                (True,  False, False): "Wrong value only",
                (False, True,  False): "Wrong colour only",
                (False, False, True ): "Wrong size only",
            }.get(wrong, "Other")

        failures.append({
            "idx":          i,
            "failure_mode": mode,
            "gt_count":     gt_n,
            "pred_count":   pred_n,
            "gt_text":      gt_sent,
            "pred_text":    pred_sent,
            "image_path":   os.path.join(
                                DATASET_ROOT, "test", "images",
                                test_rows[i]["image"]),
        })

    return failures

In [ ]:
all_failures = {}

print("=" * 70)
print("  FAILURE MODE BREAKDOWN — TEST SET")
print("=" * 70)

for model_name, result in all_results.items():
    failures = categorise_failures(result["test_preds"])
    all_failures[model_name] = failures

    total_test = len(result["test_preds"]["true_count"])
    total_err  = len(failures)
    counts     = Counter(f["failure_mode"] for f in failures)

    print(f"\n── {model_name}  ({total_err}/{total_test} errors,"
          f" {100 * total_err / total_test:.1f}% error rate) ──")
    for mode in FAILURE_ORDER:
        n = counts.get(mode, 0)
        if n:
            print(f"  {mode:<35s} {n:4d}  ({100 * n / total_err:.1f}%)")

  FAILURE MODE BREAKDOWN — TEST SET

── CustomCNN  (735/900 errors, 81.7% error rate) ──
  Wrong value only                      25  (3.4%)
  Wrong colour only                     88  (12.0%)
  Wrong size only                       22  (3.0%)
  Wrong value + colour                 111  (15.1%)
  Wrong value + size                    18  (2.4%)
  Wrong colour + size                   42  (5.7%)
  Multiple attributes wrong            429  (58.4%)

── ResNet-18  (598/900 errors, 66.4% error rate) ──
  Wrong value only                      28  (4.7%)
  Wrong colour only                     37  (6.2%)
  Wrong size only                       46  (7.7%)
  Wrong value + colour                  81  (13.5%)
  Wrong value + size                    44  (7.4%)
  Wrong colour + size                   46  (7.7%)
  Multiple attributes wrong            316  (52.8%)

── EfficientNet-B0  (570/900 errors, 63.3% error rate) ──
  Wrong value only                      39  (6.8%)
  Wrong colour only          

In [ ]:
fig, axes = plt.subplots(1, len(all_results),
                         figsize=(6 * len(all_results), 5),
                         facecolor=BG)

if len(all_results) == 1:
    axes = [axes]

for ax, (model_name, failures) in zip(axes, all_failures.items()):
    counts = Counter(f["failure_mode"] for f in failures)
    modes  = [m for m in FAILURE_ORDER if counts.get(m, 0) > 0]
    vals   = [counts[m] for m in modes]
    colour = PALETTE[model_name]

    bars = ax.barh(modes[::-1], vals[::-1], color=colour, alpha=0.85,
                   edgecolor="#1e293b")
    ax.bar_label(bars, padding=4, color="white", fontsize=8, fontweight="bold")

    ax.set_facecolor(BG)
    for sp in ax.spines.values():
        sp.set_edgecolor("#334155")
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.xaxis.grid(True, color="#1e293b", linestyle="--", lw=0.5)
    ax.set_axisbelow(True)
    ax.set_xlabel("Error count", color=MUTED)
    ax.set_title(f"{model_name}\nFailure Modes", color=colour,
                 fontsize=10, fontweight="bold")

fig.suptitle("Failure Mode Distribution — Test Set",
             color="white", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()

fm_bar_path = OUTPUT_DIR / "failure_mode_bars.png"
fig.savefig(fm_bar_path, dpi=130, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved → {fm_bar_path}")

Saved → outputs/failure_mode_bars.png


In [ ]:
N_SAMPLES    = 3     # images per failure-mode column
IMG_W, IMG_H = 4, 4  # inches per thumbnail

for model_name, failures in all_failures.items():

    # Group failures by mode
    by_mode = {}
    for f in failures:
        by_mode.setdefault(f["failure_mode"], []).append(f)

    active_modes = [m for m in FAILURE_ORDER if m in by_mode]
    if not active_modes:
        print(f"{model_name}: no failures to display.")
        continue

    n_cols = len(active_modes)
    n_rows = N_SAMPLES

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(IMG_W * n_cols, IMG_H * n_rows + 0.8),
                             facecolor=BG)

    # Normalise to always be 2-D
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes[np.newaxis, :]
    elif n_cols == 1:
        axes = axes[:, np.newaxis]

    colour = PALETTE[model_name]

    for col_j, mode in enumerate(active_modes):
        samples = by_mode[mode].copy()
        random.shuffle(samples)           # random representatives each run
        shown = samples[:n_rows]

        axes[0, col_j].set_title(mode, color=colour,
                                 fontsize=8, fontweight="bold", pad=4)

        for row_i in range(n_rows):
            ax = axes[row_i, col_j]
            ax.set_facecolor(BG)
            for sp in ax.spines.values():
                sp.set_edgecolor("#334155")
            ax.set_xticks([]); ax.set_yticks([])

            if row_i >= len(shown):
                ax.axis("off")
                continue

            fail = shown[row_i]
            try:
                ax.imshow(Image.open(fail["image_path"]).convert("RGB"))
            except FileNotFoundError:
                ax.text(0.5, 0.5, "image\nnot found",
                        ha="center", va="center", color=MUTED,
                        fontsize=7, transform=ax.transAxes)
                continue

            ax.set_xlabel(
                f"GT:   {fail['gt_text']}\nPred: {fail['pred_text']}",
                fontsize=5.5, color=MUTED, labelpad=3,
            )

    fig.suptitle(
        f"{model_name} — Sample Failures by Mode  ({len(failures)} total errors)",
        color="white", fontsize=11, fontweight="bold", y=1.005,
    )
    plt.tight_layout()

In [75]:
# Saving the data to my google drive
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define where to save on your Drive
save_path = Path("/content/drive/MyDrive/dice_experiment")

# Remove old version if it exists (so we get a clean copy)
if save_path.exists():
    shutil.rmtree(save_path)
save_path.mkdir(parents=True, exist_ok=True)

# Copy outputs folder
shutil.copytree(OUTPUT_DIR, save_path / "outputs")
print(f"Outputs saved  {save_path / 'outputs'}")

# Copy generated dataset
shutil.copytree("/content/dataset", save_path / "dataset")
print(f"Dataset saved  {save_path / 'dataset'}")

# Copy any .png files in the root /content/ directory
for png in Path("/content").glob("*.png"):
    shutil.copy(png, save_path / png.name)
    print(f"PNG saved  {save_path / png.name}")

print("\nAll files saved to Google Drive successfully!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs saved  /content/drive/MyDrive/dice_experiment/outputs
Dataset saved  /content/drive/MyDrive/dice_experiment/dataset
PNG saved  /content/drive/MyDrive/dice_experiment/SBERT Loss Curve.png
PNG saved  /content/drive/MyDrive/dice_experiment/TF-IDF Loss Curve.png
PNG saved  /content/drive/MyDrive/dice_experiment/One-hot Loss Curve.png

All files saved to Google Drive successfully!
